# 1. Introdução

&emsp; Este notebook tem como objetivo realizar o tratamento dos dados fornecidos pela instituição EAFIT, a fim de limpar e organizar o dataset para então desenvolver uma análise dos dados, com o objetivo de fornecer informações detalhadas do problema do parceiro. A partir desses processos, será possível desenvolver um modelo ideal para suprir as necessidades da instituição. Dessa forma, ao final dessa etapa, espera-se entregar um modelo preditivo que seja capaz de identificar de forma precisa se o aluno possui risco de reprovar na matéria de pensamento computacional, para que o professor possa identificar esses alunos e então traçar um plano para que essa situação não ocorra.

- **Como executar o notebook?**
    - Instale as dependências com o comando: `pip install -r notebooks/requirements.txt`.
    - Selecione o intérprete Python do ambiente como `.venv`.
    - Certifique-se de que os arquivo com os dados do primeiro e segundo semestre estão na raiz do projeto com os seguintes nomes: `Datos_Anonimo_20231_v2.xlsx` e `Datos_Anonimo_20231_v2.xlsx`.
    - Certifique-se de que existe a pasta `dados/` sem nenhum arquivo dentro.
    - Execute as células do jupyter notebook normalmente.

## 1.1. Objetivos

&emsp; Os principais objetivos desse notebook compreende:

- **Explorar e tratar os dados**: Analisar o dataset para melhorar a qualidade dos dados que o modelo consumirá.
- **Validar suposições**: Avaliar se as suposições iniciais acerca do desempenho dos alunos estão corretas a partir da formulação de gráficos e validação matemática.
- **Formular um modelo preditivo**: Desenvolver um modelo preditivo que seja eficaz na indentificação de alunos que estão em risco de serem reprovados.

# 2. Configuração de Ambiente

&emsp; Antes de tudo, é necessário preparar o ambiente de execução do notebook, realizando a importação das bibliotecas necessárias, além da definição de variáveis e funções que auxiliarão no processamento e padronização dos dados.

## 2.1. Importação de bibliotecas

&emsp; Na célula abaixo, importou-se todas as bibliotecas necessárias para realizar o processamento e análise dados, além do teste e treino do modelo. As principais bibliotecas utilizadas foram: numpy para manipulação de arrays e matrizes, seaborn e matplotlib para a visualização de dados, pandas para manipulação da planilha de dados, plotly para visualização interativa dos dados, e o scikit-learn para o processo de machine learning.

In [ ]:
# Biblioteca dedicada para encontrar padrões em strings
import re

# Bibliotecas dedicadas a manipulação matemática do projeto
import math # Permite o acesso a funções matemáticas
import numpy as np # Permite a manipulação de arrays e matrizes multidimensionais


# Bibliotecas dedicadas a manipulação do dataset e apresentação dos dados
import pandas as pd # Permite a manipulação da planilhas com os dados fornecidos pelo parceiro
import seaborn as sns # Permite a criação de gráficos usando matplotlib
import matplotlib.pyplot as plt # Dedicada para plotagem e criação de visualizações de dados
import plotly.graph_objects as go # Dedicada a criação de visualizações de dados interativas
from IPython.display import display # Permite um ambiente de uso do python de maneira interativva
from plotly.subplots import make_subplots # Permite a criação de juntar graficos em um output
import plotly.io as pio # Permite a personalização de baixo nível para as figuras feitas usando plotly
pio.renderers.default = "vscode" # Define o ambiente de renderização da visualização interativa


# Bibliotecas dedicadas ao treino e teste do modelo
from sklearn.model_selection import train_test_split, GridSearchCV # Ferramenta para a selação de dados e hiperparâmetros que serão usados pelo modelo
from imblearn.over_sampling import SMOTE # Adiciona o uso do smote
from sklearn.neighbors import NearestCentroid # Adiciona o modelo que será usado
from sklearn.metrics import recall_score, ConfusionMatrixDisplay, confusion_matrix, f1_score # Dedicada ao fornecimento de métricas para o resultado do modelo
from sklearn.preprocessing import MinMaxScaler, StandardScaler # Permite o desenvolvimento do escalonamento
from scipy.stats import shapiro ## Verificar se um conjunto de dados tem um distribuição normal
import shap # Adiciona uma maneira de calcuar os valores SHAP do modelo

## 2.2. Importação do dataframe

&emsp; É importante que a planilha com os dados fornecidos pela empresa estejam associadas a uma variável para facilitar o seu uso no código. O dataset fornecido pela empresa está no formato .xlsx e separados no 1° e 2° semestres, esses arquivos estão localizados na raiz do projeto. Dessa maneira, dedicou-se uma variável com o caminho desses arquivos, além de uma variável que indica em que aba os dados estão localizados.

In [ ]:
# Caminhos de entrada
XLSX1_PATH = "../Datos_Anonimo_20231_v2.xlsx"  # 1º semestre
XLSX2_PATH = "../Datos_Anonimo_20232_v2.xlsx"  # 2º semestre
SHEET_INDEX = 1  # segunda aba

## 2.3. Mapa de tradução dos dados para português e mapeamento de erros

&emsp; A universidade EAFIT é uma instituição de colombiana de ensino superior, e todos os materiais fornecidas pela faculdade estão em espanhol. Para facilitar o processamento e análise dos dados, optou-se por traduzir as colunas durante essa fase inicial, para facilitar a compreensão dos resultados obtidos. Assim, primeiramente montou-se um dicionário com os títulos das colunas em espanhol e o seu equivalente em português.
&emsp; Além disso, montaram outros dicionários com os dados de programas, aprovação, idade e gênero em espanhol e o seu equivalente em portguês. 
&emsp; Por fim, realizou-se um conjunto com as possíveis mensagens de erros que podem estar presente na planilha, para facilitar o encontro e substituição dessas células no tratamento de dados.

In [ ]:
# Renomeação de colunas ES -> PT (sem acentos nos nomes de colunas para evitar problemas)
rename_map = {
    "Periodo": "Periodo",
    "Grupo": "Grupo",
    "Horario": "Horario",
    "Tipo_Documento": "Tipo_Documento",
    "Edad": "Idade",
    "Genero": "Genero",
    "STEM": "STEM",
    "MejoraNotaQuices": "MelhoraNotaQuizzes",
    "Calificación_Oficial": "Nota_Oficial",
    "Aprobo": "Aprovou",
    "Nombre_Programa_Academico": "Nome_Programa_Academico",
    "Nombre_Programa_Académico": "Nome_Programa_Academico",
    "Proyecto_Parte1": "Projeto_Parte1",
    "Proyecto_Parte2": "Projeto_Parte2",
    "Talleres": "Oficinas",
    "Quices": "Quizzes",
    "CalcNotaQuiz": "CalcNotaQuiz",
    "Parcial_1": "Parcial_1",
    "Parcial_2": "Parcial_2",
    "Quiz1": "Quiz1",
    "Quiz2": "Quiz2",
    "Quiz3": "Quiz3",
    "Quiz4": "Quiz4",
    "Quiz5": "Quiz5",
    "Quiz6": "Quiz6",
    "Quiz7": "Quiz7",
    "Quiz8": "Quiz8",
    "Cuánto mejora?": "Quanto_Melhora",
    "Cuanto mejora?": "Quanto_Melhora",
}

# N mínimo prático para o dataset atual: 8 (fecha/tiempo 1..8)
for i in range(1, 9):
    rename_map[f"Fecha_Quiz{i}"] = f"Data_Quiz{i}"
    rename_map[f"TiempoQ{i}"] = f"TempoQ{i}"

MAPA_PROGRAMAS = {
    "COMUNICACIÓN SOCIAL": "Comunicacao Social",
    "DERECHO": "Direito",
    "INGENIERÍA CIVIL": "Engenharia Civil",
    "ADMINISTRACIÓN DE NEGOCIOS": "Administracao de Negocios",
    "INGENIERÍA DE DISEÑO DE PRODUCTO": "Engenharia de Design de Produto",
    "MERCADEO": "Marketing",
    "PSICOLOGÍA": "Psicologia",
    "INGENIERÍA FÍSICA": "Engenharia Fisica",
    "NEGOCIOS INTERNACIONALES": "Negocios Internacionais",
    "BIOLOGÍA": "Biologia",
    "CIENCIAS POLÍTICAS": "Ciencias Politicas",
    "ECONOMÍA": "Economia",
    "CONTADURÍA PÚBLICA": "Contabilidade Publica",
    "MÚSICA": "Musica",
    "LITERATURA": "Literatura",
    "INGENIERÍA DE PROCESOS": "Engenharia de Processos",
    "CONVENIO MOVILIDAD PREGRADO (CONVENIOS - MOVILIDAD NACIONAL - ASISTENTES PREGRADO)":
        "Convenio Mobilidade Graduacao (Convenios - Mobilidade Nacional - Assistentes Graduacao)",
}
MAPA_APROV = {"Aprobó": "Aprovou", "Reprobó": "Reprovou", "Aprobo": "Aprovou", "Reprobo": "Reprovou"}
MAPA_IDADE = {"Mayor": "Maior", "Menor": "Menor"}
MAPA_GENERO = {"femenino": "Feminino", "masculino": "Masculino", "Femenino": "Feminino", "Masculino": "Masculino"}

# Padronização de STEM para PT: "Sim" / "No"
MAPA_STEM = {
    "Sí": "Sim", "SÍ": "Sim", "si": "Sim", "SIM": "Sim", "Sim": "Sim",
    "YES": "Sim", "Yes": "Sim", "TRUE": "Sim", "True": "Sim", "1": "Sim",
    "No": "No", "NO": "No", "Nao": "No", "Não": "No", "nao": "No",
    "FALSE": "No", "False": "No", "0": "No",
}

ERROR_TOKENS = {
    "#ERROR!", "#DIV/0!", "#N/A", "#NAME?", "#NULL!", "#NUM!", "#VALUE!", "#REF!",
    "N/D", "N/A", "NA", "NaN", "nan", "None", "NONE"
}

## 2.4. Definição das Funções de Pré-Processamento

&emsp; Esta seção define um conjunto de funções dedicadas ao processo de pré-processamento dos dados. O objetivo é transformar os dados brutos, que frequentemente contêm inconsistências e formatos inadequados, em um dataset limpo, estruturado e numericamente representado, pronto para a análise e o futuro treinamento do modelo. Cada função possui uma tarefa específica para cumprir este processo.

* **Aplicação dos mapas de tradução e erros:**
    * A função `replace_excel_errors()` aplica um mapa de mensagens de erro para localizá-los nas células da planilha e, então, substituí-los por valores nulos (`NaN`) para tratamento posterior.
    * A função `padroniza_df()` atua como a principal orquestradora da padronização, aplicando a renomeação de colunas, a limpeza de erros e, crucialmente, a **uniformização de diversas colunas categóricas** (como `Nome_Programa_Academico`, `Aprovou`, `Genero`, etc.) através de mapeamentos pré-definidos.

* **Tratamento dos dados de tempo:**
    * A função principal, `parse_to_seconds()`, é utilizada para converter diversos formatos de tempo (como "X minutos Y segundos" ou "MM:SS") em uma unidade numérica única: segundos. Em seguida, a função auxiliar `converte_colunas_tempo()` aplica essa conversão a todas as colunas de tempo de quiz no dataset, garantindo um formato consistente que o modelo possa interpretar.

* **Tratamento de dados que possam atrapalhar o treinamento do modelo:**
    * A função `filtra_alunos_presentes()` remove do dataset os registros de alunos que não participaram de **nenhum** quiz (ou seja, não possuem data de realização em nenhuma atividade), garantindo que a análise contenha apenas participantes que realizaram ao menos uma avaliação.
    * A função `remove_outliers()` primeiramente identifica valores extremos (outliers) através do método do Intervalo Interquartil (IQR). Após isso, é aplicado o **clipping**, que substitui o valor dos outliers pelo limite mais próximo encontrado. Esse método preserva a informação da variável enquanto reduz o impacto negativo dos valores extremos.
        * A fundamentação matemática desse processo ocorre nos seguintes passos:
            * Cálculo do primeiro quartil ($Q_1$, 25º percentil) e do terceiro quartil ($Q_3$, 75º percentil).
            * Cálculo do intervalo interquartil: $IQR = Q_3 - Q_1$.
            * Definição dos limites superior e inferior:
                * Limite Inferior: $Q_1 - 1.5 \times IQR$
                * Limite Superior: $Q_3 + 1.5 \times IQR$
            * Após isso, aplica-se o clipping utilizando uma função nativa da biblioteca Pandas.

* **Aplicação do One-Hot Encoding:**
    * Primeiramente, a função `fit_onehot_encoder()` **treina** um codificador One-Hot. Ela analisa os dados para aprender todas as categorias únicas e cria um "molde" para a transformação. O One-Hot Encoding é utilizado para converter variáveis categóricas nominais em um formato numérico, evitando a criação de uma falsa relação de ordem que ocorreria ao atribuir números sequenciais.
    * Em seguida, a função `aplica_onehot()` usa o codificador já treinado para efetivamente **transformar** os dados, aplicando o "molde" para converter as colunas de texto em suas representações numéricas binárias.

* **Obtenção de estatísticas básicas:**
    * Para obter uma primeira análise dos dados, a função `estatisticas_basicas()` gera um **resumo estatístico** das principais colunas numéricas (média, desvio padrão, quartis, etc.) utilizando o método `.describe()` do Pandas.
    

In [ ]:
# === APLICAÇÃO DOS MAPAS DE TRADUÇÃO E ERRO ===
# Limpar células com error
def replace_excel_errors(df, cols=None):
    df = df.copy()
    if cols is None:
        cols = df.select_dtypes(include="object").columns.tolist()
    for c in cols:
        if c in df.columns:
            df[c] = df[c].replace(list(ERROR_TOKENS), np.nan)
    return df

# Padronização do dataframe
def padroniza_df(df):
    df = df.copy().rename(columns=rename_map)
    df = replace_excel_errors(df)
    if "Nome_Programa_Academico" in df.columns:
        df["Nome_Programa_Academico"] = (
            df["Nome_Programa_Academico"].astype(str).str.strip().replace(MAPA_PROGRAMAS, regex=False)
        )
    if "Aprovou" in df.columns:
        df["Aprovou"] = df["Aprovou"].astype(str).str.strip().replace(MAPA_APROV, regex=False)
    if "Idade" in df.columns:
        df["Idade"] = df["Idade"].astype(str).str.strip().replace(MAPA_IDADE, regex=False)
    if "Genero" in df.columns:
        df["Genero"] = df["Genero"].astype(str).str.strip().replace(MAPA_GENERO, regex=False)
    if "STEM" in df.columns:
        mask = df["STEM"].notna()
        if mask.any():
            df.loc[mask, "STEM"] = df.loc[mask, "STEM"].astype(str).str.strip().replace(MAPA_STEM, regex=False)
    return df

# === TRATAMENTO DOS DADOS DE TEMPO ===
# Regex para parse de tempo (ex.: "2 minutos 10 segundos")
_PATTERN_TEMPO = re.compile(
    r"^\s*(?:(\d+)\s*minuto(?:s)?)?\s*(?:(\d+)\s*segundo(?:s)?)?\s*$",
    re.IGNORECASE
)
# Converte diversos formatos de tempo para segundos 
def parse_to_seconds(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)) and not isinstance(val, bool):
        try:
            return int(val)
        except Exception:
            return np.nan
    if isinstance(val, str):
        s = val.strip()
        try:
            return int(float(s))
        except Exception:
            pass
        s_norm = s.replace(" e ", " ").replace(",", " ")
        m = _PATTERN_TEMPO.match(s_norm)
        if m:
            mins, secs = m.group(1), m.group(2)
            total = (int(mins) * 60 if mins else 0) + (int(secs) if secs else 0)
            return total if (mins or secs) else np.nan
        if ":" in s_norm:
            parts = [p.strip() for p in s_norm.split(":")]
            if len(parts) == 2 and all(p.isdigit() for p in parts):
                mm, ss = map(int, parts)
                return mm * 60 + ss
            if len(parts) == 3 and all(p.isdigit() for p in parts):
                hh, mm, ss = map(int, parts)
                return hh * 3600 + mm * 60 + ss
    return np.nan

# Padroniza as colunas de tempo
def converte_colunas_tempo(df):
    df = df.copy()
    cols_tempo = [c for c in df.columns if re.match(r"^(TempoQ|TiempoQ)\d+$", c)]
    for col in cols_tempo:
        df[col] = df[col].apply(parse_to_seconds)
    return df

# === TRATAMENTO DE DADOS QUE POSSAM ATRAPALHAR A ANÁLISE ===
# Remove os alunos faltantes
def filtra_alunos_presentes(df):
    df = df.copy()
    cols_data = [c for c in df.columns if re.match(r"^(Data_Quiz|Fecha_Quiz)\d+$", c)]
    if not cols_data:
        return df
    df_filtrado = df.dropna(subset=cols_data, how="all")
    df_filtrado = df_filtrado.drop(columns=cols_data)
    return df_filtrado

# Tratar os outliers usando clipping
def remove_outliers(df, n_cols_per_row=3):
    df_numeric = df.select_dtypes(include=["number"])
    df_numeric = df_numeric.drop(columns=["Grupo"], errors="ignore")  # evita erro se não existir

    n_cols = len(df_numeric.columns)
    n_rows = math.ceil(n_cols / n_cols_per_row)

    # Normalizar os outliers (clip)
    for col in df_numeric.columns:
        Q1 = df_numeric[col].quantile(0.25)
        Q3 = df_numeric[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df_numeric[col] = df_numeric[col].clip(lower, upper)

    return df_numeric

# === APLICAÇÃO DO ONE-HOT ===
# Treina o codificador One-Hot e define a sua estrutura
def fit_onehot_encoder(df_cat, cat_cols):
    cat_cols_exist = [c for c in cat_cols if c in df_cat.columns]
    if not cat_cols_exist:
        return {"type": "none", "encoder": None, "columns": [], "cat_cols": []}
    X_fit = replace_excel_errors(df_cat[cat_cols_exist].copy(), cols=cat_cols_exist)
    X_fit = X_fit.fillna("MISSING").astype(str)
    try:
        from sklearn.preprocessing import OneHotEncoder
        try:
            enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        except TypeError:
            enc = OneHotEncoder(sparse=False, handle_unknown="ignore")
        enc.fit(X_fit)
        cols = list(enc.get_feature_names_out(cat_cols_exist))
        return {"type": "sklearn", "encoder": enc, "columns": cols, "cat_cols": cat_cols_exist}
    except Exception:
        dummies = pd.get_dummies(X_fit, prefix=cat_cols_exist, dummy_na=False)
        cols = dummies.columns.tolist()
        return {"type": "pandas", "encoder": cols, "columns": cols, "cat_cols": cat_cols_exist}

# Aplica o codificador já treinado com os dados
def aplica_onehot(df, encoder_info):
    df = df.copy()
    if encoder_info["type"] == "none":
        return df
    cat_cols_exist = encoder_info["cat_cols"]
    X = replace_excel_errors(df[cat_cols_exist].copy(), cols=cat_cols_exist).fillna("MISSING").astype(str)
    if encoder_info["type"] == "sklearn":
        arr = encoder_info["encoder"].transform(X)
        encoded_df = pd.DataFrame(arr, columns=encoder_info["columns"], index=df.index)
        out = pd.concat([df.drop(columns=cat_cols_exist), encoded_df], axis=1)
    else:
        dummies_df = pd.get_dummies(X, prefix=cat_cols_exist, dummy_na=False)
        all_cols = encoder_info["columns"]
        for c in all_cols:
            if c not in dummies_df.columns:
                dummies_df[c] = 0
        dummies_df = dummies_df[all_cols]
        out = pd.concat([df.drop(columns=cat_cols_exist), dummies_df], axis=1)
    bad_cols = [c for c in out.columns if "#ERROR!" in c]
    if bad_cols:
        out = out.drop(columns=bad_cols)
    return out

# === OBTENÇÃO DE ESTATÍSTICAS BÁSICAS ===
# Calcula as estatísticas descritivas para cada uma das colunas de interesse
def estatisticas_basicas(df):
    possiveis = [f"Quiz{i}" for i in range(1, 13)] + [f"TempoQ{i}" for i in range(1, 13)]
    possiveis += ["Oficinas", "Parcial_1", "Parcial_2", "CalcNotaQuiz", "Nota_Oficial"]
    cols_exist = [c for c in possiveis if c in df.columns]
    if not cols_exist:
        return pd.DataFrame()
    num_cols = [c for c in cols_exist if pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        return pd.DataFrame()
    return df[num_cols].describe().copy()

# 3. Processamento dos Dados


&emsp; A qualidade desta etapa é fundamental, pois o desempenho de um modelo preditivo é diretamente dependente da qualidade dos dados com os quais ele é alimentado. Um tratamento inadequado pode introduzir vieses ou ruídos, comprometendo a capacidade do algoritmo de aprender os padrões corretos.

&emsp; Para garantir a integridade dos dados, uma sequência de tratamentos será aplicada. Este processo inclui a padronização de variáveis categóricas para garantir uniformidade, a conversão de dados de tempo para uma unidade padrão em segundos, a remoção de registros irrelevantes, o tratamento de valores extremos (*outliers*) que poderiam distorcer o modelo, e a transformação de variáveis textuais em um formato numérico através do *One-Hot Encoding*. Cada uma dessas etapas será detalhada nas subseções a seguir.&emsp; Após a importação e a análise exploratória inicial, esta seção se aprofunda na etapa mais crítica de qualquer projeto de *machine learning*: o **processamento dos dados**. O objetivo aqui é transformar o conjunto de dados brutos — que naturalmente contém inconsistências, formatos variados e erros — em uma base de dados limpa, coesa e, acima de tudo, numérica, pronta para ser utilizada pelo modelo `Nearest Centroid`.


## 3.1 Pipeline de Execução do Processamento

&emsp; Com as funções de tratamento definidas na seção anterior, esta etapa executa o pipeline de pré-processamento de ponta a ponta. O objetivo é orquestrar a aplicação dessas funções na sequência correta para transformar os arquivos de dados brutos em conjuntos de treino e teste limpos, estruturados e prontos para a modelagem.

O processo segue uma ordem lógica e metodologicamente rigorosa:

1.  **Carregamento e Divisão Estratégica dos Dados:**
    * Inicialmente, os dois arquivos Excel são carregados. A etapa mais crucial ocorre em seguida: cada um desses conjuntos de dados é imediatamente dividido em subconjuntos de **treino (80%)** e **teste (20%)** usando a função `train_test_split`.
    * **Justificativa:** Realizar a divisão neste momento inicial é uma boa prática fundamental para prevenir o **vazamento de dados (data leakage)**. Isso garante que nenhuma informação do conjunto de teste influencie as etapas de limpeza, padronização ou codificação aplicadas ao conjunto de treino, tornando a avaliação final do modelo mais realista e confiável.

2.  **Limpeza e Padronização do Conjunto de Treino:**
    * As funções de limpeza (`padroniza_df`), conversão de tempo (`converte_colunas_tempo`) e filtragem de alunos (`filtra_alunos_presentes`) são aplicadas **exclusivamente aos dados de treino**.
    * Os dois dataframes de treino já processados são então unificados em um único conjunto, o `df_consolidado`.

3.  **Tratamento de Outliers:**
    * A função `remove_outliers`, que utiliza o método de *clipping* baseado no Intervalo Interquartil (IQR), é aplicada ao `df_consolidado`. Isso assegura que valores extremos no conjunto de treino sejam suavizados, evitando que eles distorçam o cálculo dos centroides na fase de modelagem.

4.  **Codificação de Variáveis Categóricas (One-Hot Encoding):**
    * O processo de transformação das variáveis textuais em formato numérico é realizado em duas fases sobre o conjunto de treino:
        * Primeiro, o codificador é **treinado (`fit_onehot_encoder`)** utilizando apenas os dados do `df_consolidado`. O modelo aprende todas as categorias possíveis a partir do conjunto de treino.
        * Em seguida, o codificador já treinado é usado para **transformar (`aplica_onehot`)** o `df_consolidado` em sua versão final, `df_onehot`.

5.  **Processamento do Conjunto de Teste:**
    * De forma análoga, os dois dataframes de teste, que foram mantidos "intocados" até agora, são unificados.
    * As mesmas funções de padronização e limpeza são então aplicadas a este conjunto para garantir que ele tenha o mesmo formato do conjunto de treino. A codificação One-Hot é realizada em seguida para prepará-lo para a avaliação.

6.  **Verificação Final:**
    * Ao final do pipeline, as dimensões dos dataframes processados são impressas e estatísticas descritivas são geradas para o conjunto de treino final, permitindo uma verificação da consistência e dos resultados de todo o processo de limpeza e transformação.

In [ ]:
# 1) Carregar Excel
df1_raw = pd.read_excel(XLSX1_PATH, sheet_name=SHEET_INDEX)
df2_raw = pd.read_excel(XLSX2_PATH, sheet_name=SHEET_INDEX)

df1_treino, df1_teste = train_test_split(df1_raw, test_size = 0.2, random_state = 42)
df2_treino, df2_teste = train_test_split(df2_raw, test_size = 0.2, random_state = 42)
# 2) Padronizar valores/nomes
df1 = padroniza_df(df1_treino)
df2 = padroniza_df(df2_treino)


# 3) Converter tempos para segundos
df1 = converte_colunas_tempo(df1)
df2 = converte_colunas_tempo(df2)


# 4) Filtrar apenas alunos com algum quiz e remover colunas de datas
df1_presentes = filtra_alunos_presentes(df1)
df2_presentes = filtra_alunos_presentes(df2)

# 5) Concatenar
df_consolidado = pd.concat([df1_presentes, df2_presentes], ignore_index=True)

df_teste = pd.concat([df1_teste, df2_teste], ignore_index=True)
df_teste = padroniza_df(df_teste)

# 5.1) Limpar outliers das colunas numéricas
df_numeric_clean = remove_outliers(df_consolidado)  # faz clip nos outliers
# Substituir as colunas numéricas no df_consolidado
df_consolidado[df_numeric_clean.columns] = df_numeric_clean

# 6) Remover linhas com STEM ausente, replicando lógica original
if "STEM" in df_consolidado.columns:
    df_consolidado = df_consolidado.dropna(subset=["STEM"])

# 7) One-Hot
cat_cols = ["Tipo_Documento", "Idade", "Genero", "STEM", "MelhoraNotaQuizzes", "Aprovou"]
encoder_info = fit_onehot_encoder(df_consolidado, cat_cols)
df_onehot = aplica_onehot(df_consolidado, encoder_info)

encoder_info_teste = fit_onehot_encoder(df_teste, cat_cols)
df_teste = aplica_onehot(df_teste, encoder_info_teste)
df_teste = converte_colunas_tempo(df_teste)
df_teste = padroniza_df(df_teste)
# 8) Estatísticas
tabela_stats = estatisticas_basicas(df_consolidado)

# Amostras
print("df_consolidado:", df_consolidado.shape)
print("df_onehot:", df_onehot.shape)


## 3.2. Refinamento Final do Dataset: Remoção de Colunas Redundantes

&emsp; Após a execução do pipeline de processamento principal, esta etapa realiza um refinamento final no dataframe `df_onehot`. O objetivo é otimizar a estrutura dos dados para a modelagem, removendo redundâncias matemáticas e garantindo uma ordem consistente para as variáveis.

* **Remoção de *Baselines* para Evitar a Multicollinearidade:**
    * A primeira ação é remover um conjunto específico de colunas, como `Genero_Feminino`, `STEM_No`, entre outras. Estas colunas são consideradas a "categoria de base" (*baseline*) de suas respectivas variáveis originais.
    * **Justificativa Técnica:** Esta remoção é uma prática padrão para evitar a **"armadilha da variável dummy" (*dummy variable trap*)**. Quando o *One-Hot Encoding* converte uma variável com *k* categorias em *k* novas colunas, a informação se torna redundante (ex: se um aluno não é `Genero_Masculino=1`, ele necessariamente é `Genero_Feminino=1`). Essa redundância, chamada de **multicollinearidade perfeita**, pode causar problemas em alguns algoritmos de machine learning. Ao remover uma categoria de base para cada variável, eliminamos a redundância, mantendo 100% da informação original.

* **Reordenação das Colunas:**
    * A segunda e última ação é organizar as colunas restantes em uma ordem explícita e predefinida.
    * **Justificativa Técnica:** Embora a ordem das colunas não altere o conteúdo do dataframe, fixá-la é crucial para a **consistência e reprodutibilidade** do projeto. Isso garante que o modelo sempre receberá os dados na mesma estrutura, evitando erros e facilitando a análise de importância das variáveis na fase de interpretação dos resultados.

&emsp; Ao final desta etapa, o dataframe `df_onehot` está em seu formato definitivo, otimizado e pronto para ser utilizado no treinamento do modelo `Nearest Centroid`.

In [ ]:
# Remoção segura: só dropa se existir
baselines = ["STEM_No", "MelhoraNotaQuizzes_False", "Idade_Menor", "Aprovou_Reprovou", 'Genero_Feminino']
drop_list = [c for c in baselines if c in df_onehot.columns]
df_onehot = df_onehot.drop(columns=drop_list)
print("Dummies baseline removidas:", drop_list)
print("df_onehot (após remoção):", df_onehot.shape)

df_onehot = df_onehot[['Tipo_Documento_CC','Tipo_Documento_CE','Tipo_Documento_PP','Tipo_Documento_TI',
                       'Idade_Maior', 'Genero_Masculino', 'STEM_SI', 'MelhoraNotaQuizzes_True', 'Aprovou_Aprovou']]


## 3.3. Estatísticas Descritivas das Variáveis de Interesse

&emsp; Após a conclusão do pipeline de limpeza e transformação, esta etapa realiza uma **análise estatística descritiva** sobre as principais variáveis numéricas do conjunto de dados de treino (`df_consolidado`). O objetivo é duplo: servir como uma **validação final** da qualidade do processamento e fornecer uma visão quantitativa sobre as características e a distribuição dos dados que alimentarão o modelo.

* **Seleção das Variáveis de Desempenho:**
    * Primeiramente, é definido um subconjunto de colunas que representam diretamente o desempenho dos alunos, como notas de quizzes, tempos de resposta, projetos e avaliações parciais. O foco nestas variáveis permite uma análise direcionada aos fatores mais relevantes para a previsão de risco.

* **Geração da Tabela Descritiva:**
    * Utiliza-se o método `.describe()` da biblioteca Pandas para calcular um resumo das principais medidas estatísticas de cada variável selecionada. A tabela resultante é transposta (`.T`) para melhorar a legibilidade, organizando as variáveis como linhas.

* **Interpretação das Métricas:**
    * A tabela gerada oferece insights valiosos através de suas colunas:
        * **`count`**: Mostra o número de registros não nulos, permitindo verificar a completude dos dados para cada variável.
        * **`mean`**: A média, que indica o valor central ou o desempenho geral dos alunos em cada atividade.
        * **`std`**: O desvio padrão, que mede a dispersão ou variabilidade dos dados. Um valor alto indica grande variação no desempenho dos alunos, enquanto um valor baixo sugere maior consistência.
        * **`min`, `max`**: Os valores mínimo e máximo, úteis para confirmar se os dados estão dentro dos intervalos esperados (ex: notas de 0 a 10) e validar a eficácia do tratamento de outliers.
        * **`25%`, `50%`, `75%`**: Os quartis, que ajudam a entender a distribuição dos dados e a performance da maior parte da turma, sendo menos sensíveis a valores extremos.

&emsp; Esta análise quantitativa finaliza a etapa de pré-processamento, fornecendo um panorama claro e validado das características do dataset que será utilizado para treinar e avaliar o modelo `Nearest Centroid`.

In [ ]:
# seleciona todas as colunas de interesse
colunas = [
    "Quiz1", "TempoQ1", "Quiz2", "TempoQ2", "Quiz3", "TempoQ3", 
    "Quiz4", "TempoQ4", "Quiz5", "TempoQ5", "Quiz6", "TempoQ6", 
    "Quiz7", "TempoQ7", "Quiz8", "Nota_Oficial", 
    "Projeto_Parte1", "Projeto_Parte2", "Quanto_Melhora", "Oficinas", "Parcial_1",
    "Parcial_2"
]

# gera tabela descritiva
desc_tabela = df_consolidado[colunas].describe().T  # Transpõe para variáveis ficarem como linhas



## 3.4. Escalonamento das Variáveis (Feature Scaling)

&emsp; Uma etapa de pré-processamento fundamental para o bom funcionamento de algoritmos baseados em distância, como o **`Nearest Centroid`**, é o **escalonamento de variáveis**. O objetivo desta célula é colocar as variáveis numéricas em uma escala comum, garantindo que nenhuma delas domine o cálculo de distância do modelo apenas por ter uma magnitude maior.

&emsp; Sem o escalonamento, uma variável com um intervalo amplo (ex: uma nota de 0 a 1000) teria um peso desproporcionalmente maior na determinação da "proximidade" a um centroide do que uma variável com um intervalo pequeno (ex: uma nota de 0 a 10). Isso distorceria o modelo, fazendo-o ignorar a influência das variáveis de menor escala. Para evitar esse problema, foram aplicadas duas técnicas de escalonamento distintas:

* **Padronização com `StandardScaler` (Z-Score):**
    * **Aplicação:** Este método foi utilizado nas colunas `Nota_Oficial` e `CalcNotaQuiz`.
    * **Fundamento Matemático:** A padronização transforma os dados de modo que eles passem a ter uma **média (μ) de 0** e um **desvio padrão (σ) de 1**. A fórmula para cada valor *x* é:
        $$ z = (x - \mu) / \sigma $$
    * **Justificativa:** Esta técnica é uma escolha robusta e muito comum, pois centraliza os dados em torno de zero e não os comprime em um intervalo fixo, o que a torna menos sensível a outliers.

* **Normalização com `MinMaxScaler` (Intervalo [0, 1]):**
    * **Aplicação:** Este método foi utilizado na coluna `Oficinas`.
    * **Fundamento Matemático:** A normalização redimensiona os dados para que se encaixem em um intervalo predefinido, neste caso, entre 0 e 1. A fórmula para cada valor *X* é:
        $$ X_{scaled} = (X - X_{min}) / (X_{max} - X_{min}) $$
    * **Justificativa:** É útil quando se deseja ter os dados em uma escala limitada e funciona bem para variáveis que não seguem necessariamente uma distribuição normal.

&emsp; Ao final deste processo, as principais variáveis numéricas do `df_scaled` estão em escalas comparáveis, garantindo que todas contribuirão de forma equitativa para o cálculo de distância do `Nearest Centroid` e, consequentemente, para uma predição mais justa e precisa.

In [ ]:
# Escalonamento das variáveis testadas

# Criando cópia para não sobrescrever
df_scaled = df_consolidado.copy()

# ----------------------
# Padronização (Z-score)
# ----------------------
scaler_standard = StandardScaler()

# Calificacion_Oficial
df_scaled["Nota_Oficial"] = scaler_standard.fit_transform(
    df_consolidado[["Nota_Oficial"]]
)

# CalcNotaQuiz
df_scaled["CalcNotaQuiz_scaled"] = scaler_standard.fit_transform(
    df_consolidado[["CalcNotaQuiz"]]
)

# ----------------------
# Min-Max [0,1]
# ----------------------
scaler_minmax = MinMaxScaler()

# Talleres
df_scaled["Oficinas"] = scaler_minmax.fit_transform(
    df_consolidado[["Oficinas"]]
)


Histograma e Violin Plot [CalcNotaQuiz]

## 3.5. Escalonamento em Lote das Variáveis de Desempenho

&emsp; Após o escalonamento individual de algumas variáveis na etapa anterior, esta seção finaliza o pré-processamento aplicando uma normalização consistente a todas as variáveis de desempenho restantes. O objetivo é garantir que todas as notas de quizzes, avaliações parciais e os respectivos tempos de resolução estejam na mesma escala numérica [0, 1], um pré-requisito fundamental para o modelo `Nearest Centroid`.

&emsp; O processo foi dividido em duas etapas, uma para cada conjunto de dados:

* **Normalização do Conjunto de Treino:**
    * Um laço de repetição (`for`) é utilizado para aplicar o `MinMaxScaler` de forma sistemática a todas as colunas de quizzes e tempos no dataframe de treino (`df_consolidado`).
    * Ao final, é criado o dataframe `df_scaled` definitivo, que contém apenas as colunas de interesse já devidamente limpas, tratadas e escalonadas, prontas para serem usadas no treinamento do modelo.

* **Normalização do Conjunto de Teste:**
    * O mesmo procedimento é então replicado para o conjunto de teste (`df_teste`), garantindo que ele tenha a mesma estrutura e escala do conjunto de treino.
    * **Nota Metodológica Importante:** O código atual utiliza `fit_transform` novamente no conjunto de teste. É importante notar que a prática mais rigorosa em pipelines de machine learning é **treinar (`fit`) o scaler *apenas* com os dados de treino** e, em seguida, usar esse **mesmo scaler já treinado para aplicar a transformação (`transform`)** nos dados de teste. Isso garante que a mesma e exata escala (os mesmos valores de mínimo e máximo aprendidos no treino) seja aplicada a ambos os conjuntos, evitando um leve vazamento de dados (*data leakage*). Para este projeto, o impacto é mínimo, mas é uma distinção conceitual crucial.

&emsp; Com a execução destas células, os dataframes de treino e teste estão totalmente processados e prontos para a etapa final de modelagem.

In [ ]:
quizzes = ['Quiz1','Quiz2','Quiz3','Quiz4','Quiz5','Quiz6','Quiz7', "Parcial_1", "Parcial_2"]

for q in quizzes:
    df_scaled[q] = scaler_minmax.fit_transform(
        df_consolidado[[q]]
    )

tempo = ['TempoQ1','TempoQ2','TempoQ3','TempoQ4','TempoQ5','TempoQ6','TempoQ7']

for t in tempo:
    df_scaled[t] = scaler_minmax.fit_transform(
        df_consolidado[[t]]
    )

#Seleciona apenas as variáveis escalonadas para que depois se crie o df final
df_scaled = df_scaled[['Quiz1','Quiz2','Quiz3','Quiz4','Quiz5','Quiz6','Quiz7','TempoQ1','TempoQ2','TempoQ3',
                       'TempoQ4','TempoQ5','TempoQ6','TempoQ7',"Nota_Oficial","CalcNotaQuiz_scaled","Oficinas","Projeto_Parte1", "Projeto_Parte2","Parcial_1",
    "Parcial_2"]]

In [ ]:
quizzes = ['Quiz1','Quiz2','Quiz3','Quiz4','Quiz5','Quiz6','Quiz7', "Parcial_1", "Parcial_2"]

for q in quizzes:
    df_teste[q] = scaler_minmax.fit_transform(
        df_teste[[q]]
    )

tempo = ['TempoQ1','TempoQ2','TempoQ3','TempoQ4','TempoQ5','TempoQ6','TempoQ7']

for t in tempo:
    df_teste[t] = scaler_minmax.fit_transform(
        df_teste[[t]]
    )



## 3.6. Montagem dos Datasets Finais para Análise Temporal

&emsp; Esta é a etapa final da preparação dos dados. O objetivo aqui é consolidar todas as features processadas e, em seguida, segmentá-las em conjuntos de dados específicos que simulam a disponibilidade de informação em cada um dos três momentos de análise do projeto (Semanas 6, 8 e 12).

O processo é executado na seguinte ordem:

1.  **Consolidação das Features Processadas:**
    * Os dataframes contendo as variáveis numéricas escaladas (`df_scaled`) e as variáveis categóricas codificadas (`df_onehot`) são concatenados para criar um dataframe mestre, `df_final`, que contém todas as features prontas para a modelagem.

2.  **Remoção de Variáveis Inadequadas:**
    * Algumas colunas são removidas dos dataframes de treino e teste. A mais crítica é a **`Nota_Oficial`**.
    * **Justificativa Técnica:** A remoção da `Nota_Oficial` é essencial para evitar o **vazamento de alvo (*target leakage*)**. Como esta variável é, ou está diretamente relacionada, ao resultado final que queremos prever, mantê-la como uma feature daria ao modelo uma "resposta pronta", resultando em uma performance artificialmente perfeita, mas inútil na prática. As colunas `Tipo_Documento_*` também são removidas para simplificar o modelo.

3.  **Criação da Variável Alvo Final (`Reprovou`):**
    * A coluna `Aprovou_Aprovou`, que veio do One-Hot Encoding, é utilizada para criar a variável alvo final, `Reprovou`. A lógica é invertida para que `1` represente o evento de interesse (reprovação), alinhando os dados de forma mais intuitiva com o objetivo do projeto. A coluna original `Aprovou_Aprovou` é então descartada.

4.  **Criação dos "Snapshots" Temporais:**
    * Este é o passo crucial que viabiliza a análise temporal. O dataframe final é subsetado para criar três conjuntos de dados distintos para treino e três para teste, cada um representando a informação disponível em um ponto específico do tempo:
        * **`df_final1` / `df_teste1`**: Contém apenas as features que estariam disponíveis na **Semana 6** (ex: até o Quiz 3).
        * **`df_final2` / `df_teste2`**: Adiciona as features disponíveis até a **Semana 8** (ex: até a Parcial 1).
        * **`df_final3` / `df_teste3`**: Utiliza o conjunto de features mais completo, simulando a **Semana 12**.
    * **Justificativa:** Esta separação rigorosa é o que garante a integridade da análise, assegurando que cada modelo seja treinado e avaliado apenas com os dados que seriam conhecidos naquele exato momento do semestre, sem "olhar para o futuro".

&emsp; Com estes seis dataframes finais gerados, a fase de pré-processamento está concluída, e o notebook está pronto para iniciar o treinamento e a avaliação sequencial do modelo `Nearest Centroid` em cada um dos três cenários definidos.

In [ ]:
# Concatenando df escalado e df onehot
df_final = pd.concat([df_scaled, df_onehot], axis=1)
colunas_para_remover = ['Nota_Oficial', 'Tipo_Documento_CC','Tipo_Documento_CE','Tipo_Documento_PP','Tipo_Documento_TI']

# Removendo as colunas
df_final = df_final.drop(columns=colunas_para_remover)

df_teste = df_teste.drop(columns=colunas_para_remover)
# df_teste = pd.concat([df_scaled_teste, df_teste], axis =1)

In [ ]:
df_final["Reprovou"] = df_final["Aprovou_Aprovou"].apply(lambda x: 0 if x == 1 else 1)
df_final.drop('Aprovou_Aprovou', axis  = 1, inplace = True)

df_teste["Reprovou"] = df_teste["Aprovou_Aprovou"].apply(lambda x: 0 if x == 1 else 1)
df_teste.drop('Aprovou_Aprovou', axis  = 1, inplace = True)

In [ ]:
df_final1 = df_final[['Genero_Masculino','STEM_SI',  'Quiz1', 'Quiz2', 'Quiz3', 'TempoQ1', 'TempoQ2', 'TempoQ3', 'Reprovou']]

df_final2 = df_final[['Genero_Masculino','STEM_SI', 'Quiz1', 'Quiz2', 'Quiz3','Quiz4', 'TempoQ1', 'TempoQ2', 'TempoQ3','TempoQ4', 'Parcial_1', 'Reprovou']]

df_final3 = df_final[['Genero_Masculino','STEM_SI', 'Quiz1', 'Quiz2', 'Quiz3','Quiz4','Quiz5', 'Quiz6', 'TempoQ1', 'TempoQ2', 'TempoQ3','TempoQ4', 'TempoQ5', 'TempoQ6','Parcial_1' , 'Reprovou']]

In [ ]:
df_teste1 = df_teste[['Genero_Masculino','STEM_SI',  'Quiz1', 'Quiz2', 'Quiz3', 'TempoQ1', 'TempoQ2', 'TempoQ3', 'Reprovou']]

df_teste2 = df_teste[['Genero_Masculino','STEM_SI', 'Quiz1', 'Quiz2', 'Quiz3','Quiz4', 'TempoQ1', 'TempoQ2', 'TempoQ3','TempoQ4', 'Parcial_1', 'Reprovou']]

df_teste3 = df_teste[['Genero_Masculino','STEM_SI', 'Quiz1', 'Quiz2', 'Quiz3','Quiz4','Quiz5', 'Quiz6', 'TempoQ1', 'TempoQ2', 'TempoQ3','TempoQ4', 'TempoQ5', 'TempoQ6','Parcial_1' , 'Reprovou']]

# 4. Exportação dos Datasets Processados

&emsp; Esta é a etapa final do notebook de pré-processamento. O objetivo aqui é salvar os dataframes gerados em arquivos no formato `.csv`. Esta prática é fundamental para garantir a **eficiência e a reprodutibilidade** do projeto, criando "checkpoints" dos dados em diferentes estágios do tratamento.

## 4.1. Geração dos Arquivos .csv

&emsp; O código a seguir exporta múltiplos dataframes para a pasta `dados/`. Cada arquivo representa um estágio específico do pipeline, permitindo que futuras análises ou o próprio notebook de modelagem possam carregar os dados já processados sem a necessidade de executar novamente todo o pipeline desde o início.

* **Arquivos de Etapas Intermediárias:**
    * São salvos dataframes como `dados_consolidado.csv` e `dados_onehot.csv`.
    * **Justificativa:** Armazenar esses resultados intermediários permite uma fácil verificação e depuração de cada fase do processo, além de modularizar o projeto.

* **Arquivos Finais para Modelagem Temporal:**
    * Os arquivos mais importantes gerados são os seis datasets de "snapshot": `dados_modelo1.csv`, `dados_modelo2.csv`, `dados_modelo3.csv` (para treino) e seus respectivos conjuntos de teste (`dados_teste1.csv`, etc.).
    * **Justificativa:** Estes são os insumos diretos para o notebook de modelagem. Salvá-los em disco garante que os experimentos e a avaliação final do modelo serão sempre executados sobre o mesmo conjunto de dados, um pilar para a reprodutibilidade dos resultados.

* **Nota Técnica sobre o Parâmetro `index=False`:**
    * Em todas as exportações, o parâmetro `index=False` é utilizado para instruir a biblioteca Pandas a não salvar o índice do dataframe como uma coluna no arquivo .csv. Isso evita a criação de colunas desnecessárias ("Unnamed: 0") que podem causar problemas ao recarregar os dados.

&emsp; Com a conclusão desta etapa, o fluxo de trabalho de preparação de dados está finalizado, e os ativos de dados estão prontos para serem consumidos na fase de treinamento e avaliação do modelo.

In [ ]:

df1.to_csv('dados/dados_bruto_sem1.csv', index=False)
df2.to_csv('dados/dados_bruto_sem2.csv', index=False)
df_consolidado.to_csv('dados/dados_consolidado.csv', index=False)
df_onehot.to_csv('dados/dados_onehot.csv', index=False)
df_scaled.to_csv('dados/dados_escalado.csv', index=False)
df_final.to_csv('dados/dados_modelo.csv', index=False)

df_final1.to_csv('dados/dados_modelo1.csv', index=False)
df_final2.to_csv('dados/dados_modelo2.csv', index=False)
df_final3.to_csv('dados/dados_modelo3.csv', index=False)

df_teste1.to_csv('dados/dados_teste1.csv', index=False)
df_teste2.to_csv('dados/dados_teste2.csv', index=False)
df_teste3.to_csv('dados/dados_teste3.csv', index=False)



## 4.2. Verificação Visual dos Dataframes Gerados

&emsp; Como etapa final de validação de todo o pipeline de pré-processamento, esta seção apresenta uma amostra das primeiras cinco linhas de cada um dos principais dataframes intermediários que foram criados e exportados. O objetivo é realizar uma **inspeção visual rápida (*sanity check*)** para confirmar que as transformações foram aplicadas corretamente em cada fase e que os dados estão no formato esperado antes de concluir o notebook.

* **`df1.head()` e `df2.head()`**:
    * Exibem os dados brutos de cada um dos dois semestres após a aplicação da padronização inicial (`padroniza_df`). Serve para confirmar que a renomeação de colunas e a limpeza básica de valores foram bem-sucedidas.

* **`df_consolidado.head()`**:
    * Mostra o resultado da unificação dos dois dataframes de treino. Esta visualização é útil para verificar se a concatenação foi realizada corretamente, resultando em um único conjunto de dados coeso.

* **`df_onehot.head()`**:
    * Permite inspecionar o resultado do *One-Hot Encoding*. A visualização deve mostrar que as colunas categóricas originais (como `Genero`, `STEM`, etc.) foram substituídas pelas novas colunas binárias (com valores 0 ou 1).

* **`df_scaled.head()`**:
    * Apresenta o dataframe após a aplicação do escalonamento (`MinMaxScaler` e `StandardScaler`). A inspeção visual aqui deve confirmar que as colunas numéricas (como `Quiz1`, `TempoQ1`, `Nota_Oficial`, etc.) agora contêm valores em uma escala comum e pequena (ex: entre 0 e 1), como esperado.

&emsp; Com esta verificação visual, o notebook de preparação de dados é concluído, e os arquivos `.csv` são salvos com a confiança de que o processamento foi executado conforme o planejado.

In [ ]:
df1.head()

In [ ]:
df2.head()

In [ ]:
df_consolidado.head()

In [ ]:
df_onehot.head()

In [ ]:
df_scaled.head()

## 4.3. Preparação dos Dados para a Modelagem

&emsp; Com os datasets devidamente processados e exportados no notebook anterior, esta seção realiza as etapas finais para preparar os dados para o treinamento e teste do modelo `Nearest Centroid`. O processo consiste em carregar os arquivos, separar as variáveis preditoras da variável alvo e garantir que não haja valores ausentes.

As etapas executadas nas células de código a seguir são:

* **1. Carregamento dos Datasets Processados:**
    * A primeira célula carrega os seis arquivos `.csv` que foram gerados ao final do pipeline de processamento. Estes arquivos representam os "snapshots" de dados para cada um dos três momentos de análise (Semanas 6, 8 e 12), para ambos os conjuntos de treino e teste.

* **2. Separação de Features (X) e Alvo (y):**
    * Para treinar um modelo de aprendizado supervisionado, é necessário separar os dados em duas partes distintas:
        * **Features (X)**: O conjunto de variáveis preditoras (as "pistas") que o modelo utilizará para aprender os padrões.
        * **Alvo (y)**: A variável que queremos prever (a "resposta"), que neste caso é a coluna `Reprovou`.
    * Este processo é repetido para cada um dos seis dataframes, criando os pares `X_train`/`y_train` e `X_test`/`y_test` para cada semana de análise.

* **3. Tratamento Final de Valores Ausentes:**
    * Como última etapa de limpeza, a célula final aplica uma técnica de **imputação por constante**, preenchendo com o valor `0` qualquer dado faltante na coluna `TempoQ3`. Isso garante que o dataset esteja numericamente completo antes do treinamento, já que algoritmos de machine learning não conseguem processar valores ausentes (`NaN`).

In [ ]:
df1 = pd.read_csv('dados/dados_modelo1.csv')
df2 = pd.read_csv('dados/dados_modelo2.csv')
df3 = pd.read_csv('dados/dados_modelo3.csv')
df1_test = pd.read_csv('dados/dados_teste1.csv')
df2_test = pd.read_csv('dados/dados_teste2.csv')
df3_test = pd.read_csv('dados/dados_teste3.csv')

In [ ]:
X1_train = df1.drop('Reprovou', axis = 1)
y1_train = df1['Reprovou']

X2_train = df2.drop('Reprovou', axis = 1)
y2_train = df2['Reprovou']

X3_train = df3.drop('Reprovou', axis = 1)
y3_train = df3['Reprovou']

X1_test = df1_test.drop('Reprovou', axis = 1)
y1_test = df1_test['Reprovou']

X2_test = df2_test.drop('Reprovou', axis = 1)
y2_test = df2_test['Reprovou']

X3_test = df3_test.drop('Reprovou', axis = 1)
y3_test = df3_test['Reprovou']

In [ ]:
X1_train['TempoQ3'] = X1_train['TempoQ3'].fillna(0)
X2_train['TempoQ3'] = X2_train['TempoQ3'].fillna(0)
X3_train['TempoQ3'] = X3_train['TempoQ3'].fillna(0)

# 5. Análise dos dados

Este notebook realiza uma análise exploratória e visual do desempenho acadêmico de alunos, consolidando dados de diferentes semestres e conjuntos (bruto, consolidado, one-hot e escalonado). Os objetivos principais são:

- Explorar relações entre variáveis de desempenho (quizzes, tempo, projetos, oficinas, parciais e nota oficial).

- Avaliar distribuições, médias, medianas, assimetria e normalidade.

- Comparar resultados por grupos (STEM vs. não-STEM) e por gênero.

- Visualizar o efeito do escalonamento/padronização (Z-score) em variáveis de interesse.

## 5.1. Importação dos Dados para Análise

&emsp; A primeira etapa da análise exploratória consiste no carregamento dos conjuntos de dados que foram processados e salvos no notebook anterior. Para esta análise, serão utilizados os arquivos que representam os dados brutos de cada semestre, já com a padronização inicial de nomes de colunas e valores aplicada.

&emsp; O código a seguir importa os seguintes arquivos `.csv`:

* **`dados_bruto_sem1.csv`**: Contém os dados do primeiro semestre, carregados no dataframe `df1_analise`.
* **`dados_bruto_sem2.csv`**: Contém os dados do segundo semestre, carregados no dataframe `df2_analise`.

&emsp; Carregar os dados de cada semestre em dataframes separados nesta fase inicial permite uma análise comparativa entre os dois períodos, antes de qualquer consolidação ou transformação mais profunda.

In [ ]:
df1_analise = pd.read_csv('dados/dados_bruto_sem1.csv')
df2_analise = pd.read_csv('dados/dados_bruto_sem2.csv')


## 5.2. Análise de Correlação entre Variáveis Numéricas

&emsp; Uma etapa fundamental na análise exploratória de dados é entender como as variáveis se relacionam entre si. Esta seção tem como objetivo quantificar e visualizar as **relações lineares** entre as variáveis numéricas do desempenho dos alunos, como notas, tempos e projetos. Para isso, utilizamos uma **matriz de correlação**, visualizada através de um **heatmap**.

&emsp; A análise é conduzida de forma independente para cada semestre (`df1_analise` e `df2_analise`), permitindo não apenas identificar as relações dentro de cada grupo, but also comparar se esses padrões se mantêm consistentes entre os dois períodos.

O processo para gerar cada gráfico segue quatro etapas principais:

* **1. Seleção de Variáveis Numéricas:**
    * Primeiramente, são selecionadas apenas as colunas do tipo numérico de cada dataframe. A coluna `Grupo`, apesar de numérica, é removida por ser um identificador e não uma variável de desempenho.

* **2. Cálculo da Matriz de Correlação:**
    * O método `.corr()` do Pandas é utilizado para calcular o **coeficiente de correlação de Pearson** entre cada par de variáveis. Este coeficiente é um valor entre -1 e 1 que indica:
        * **+1:** Correlação positiva perfeita (quando uma variável aumenta, a outra também aumenta).
        * **-1:** Correlação negativa perfeita (quando uma variável aumenta, a outra diminui).
        * **0:** Nenhuma correlação linear.

* **3. Criação da Máscara para Otimização Visual:**
    * Uma máscara é criada para ocultar a parte triangular superior do heatmap. Como a matriz de correlação é simétrica (a correlação entre A e B é a mesma que entre B e A), essa técnica remove a informação redundante, tornando o gráfico mais limpo e fácil de interpretar.

* **4. Geração do Heatmap:**
    * A biblioteca `Seaborn` é usada para plotar a matriz como um mapa de calor (*heatmap*). A paleta de cores `coolwarm` é ideal para esta visualização, pois usa tons quentes (vermelho) para correlações positivas, tons frios (azul) para correlações negativas e uma cor neutra para valores próximos de zero. Os valores numéricos são anotados em cada célula para permitir uma análise precisa.

&emsp; Ao analisar os gráficos, o foco é identificar padrões, como a esperada correlação positiva entre as notas dos diferentes quizzes ou possíveis correlações negativas entre o tempo gasto em uma tarefa e a nota obtida.

In [ ]:
#Identificando colunas numéricas
numerical_cols1 = df1_analise.select_dtypes(include = 'number').columns
numerical_cols2 = df2_analise.select_dtypes(include = 'number').columns

numerical_cols1 = numerical_cols1.drop('Grupo')
numerical_cols2 = numerical_cols2.drop('Grupo')

#crianco a tabela de correlação
correlation1 = df1_analise[numerical_cols1].corr()
correlation2 = df2_analise[numerical_cols2].corr()


# tirando a parte desnecessária que estará no heatmap
mask1 = np.triu(np.ones_like(correlation1, dtype=bool))
mask2 = np.triu(np.ones_like(correlation2, dtype=bool))

# plotando os heatmaps
fig1, ax1 = plt.subplots(figsize=(12, 10))
sns.heatmap(correlation1, annot=True, fmt='.1f', mask=mask1, square=True, linewidths=.5, ax=ax1, cmap='coolwarm')

fig2, ax2 = plt.subplots(figsize=(12, 10))
sns.heatmap(correlation2, annot=True, fmt='.1f', mask=mask1, square=True, linewidths=.5, ax=ax2, cmap='coolwarm')

#### Interpretação dos Heatmaps

&emsp; A visualização das matrizes de correlação através de *heatmaps* permite uma rápida identificação de padrões e da força das relações lineares entre as variáveis numéricas de desempenho. A análise dos gráficos gerados para os dois semestres revela os seguintes pontos principais:

* **Consistência entre os Semestres:** A primeira observação notável é a alta similaridade entre os dois *heatmaps*. Os padrões de correlação são muito consistentes em ambos os períodos, sugerindo que as relações entre as variáveis de desempenho são estáveis e não mudam drasticamente de uma turma para a outra.

* **Fortes Correlações Positivas (Tons Quentes):**
    * Conforme esperado, a `Nota_Oficial` apresenta uma forte correlação positiva com as principais avaliações do curso, como `Parcial_1` (0.7), `Parcial_2` (0.8) e as duas partes do `Projeto` (0.8). Isso confirma que o bom desempenho nessas atividades é um forte indicador de uma nota final alta.
    * Uma correlação excepcionalmente forte é visível entre `Oficinas` e `CalcNotaQuiz` (0.9), indicando que estas duas métricas estão medindo aspectos muito semelhantes do desempenho dos alunos.

* **Correlações Negativas Relevantes (Tons Frios):**
    * O insight mais interessante surge das correlações negativas. Destaca-se a forte correlação negativa entre `Quiz5` e `TempoQ5` (-0.8). Isso indica que, para essa avaliação específica, os alunos que levaram mais tempo para responder tenderam a obter as notas mais baixas, sugerindo que a rapidez e a eficiência foram cruciais para o sucesso.
    * De forma geral, observa-se uma tendência de correlações fracas a moderadas entre os tempos de execução dos quizzes e as notas finais, sugerindo que a eficiência geral é um fator associado ao bom desempenho.

&emsp; Em resumo, a análise de correlação valida a estrutura do dataset, confirmando relações esperadas, e também revela dinâmicas importantes, como a relação inversa entre tempo e desempenho em certas atividades, fornecendo um entendimento mais profundo dos fatores associados ao sucesso acadêmico.

## 5.3. Análise Comparativa de Desempenho por Grupo (STEM vs. Não-STEM)

&emsp; Esta seção da análise exploratória investiga se existe uma relação entre a área de estudo do aluno (pertencente ou não a um curso de STEM - *Science, Technology, Engineering, and Mathematics*) e o seu resultado final de aprovação ou reprovação. O objetivo é identificar se algum dos grupos apresenta uma disparidade significativa no desempenho.

O processo para criar a visualização comparativa é dividido nas seguintes etapas:

* **1. Agregação de Dados por Grupo:**
    * Primeiramente, os dados do dataframe `df_onehot` são agrupados pela variável `STEM_SI`.
    * Dentro de cada grupo (alunos de STEM e não-STEM), o método `.value_counts()` é utilizado para contar a quantidade de alunos aprovados e reprovados.
    * A função `.unstack()` é aplicada em seguida para transformar os dados em uma tabela-resumo, onde as linhas representam os grupos e as colunas contêm as contagens de aprovados e reprovados, facilitando o acesso aos dados para a plotagem.

* **2. Preparação para a Visualização:**
    * Os valores numéricos da tabela-resumo (contagens de aprovados e reprovados) e os rótulos de cada grupo são extraídos para variáveis separadas, preparando os insumos necessários para a biblioteca `Matplotlib`.

* **3. Geração do Gráfico de Barras Agrupado:**
    * **Justificativa da Escolha:** Um gráfico de barras agrupado foi selecionado por ser a visualização ideal para comparar diretamente as contagens de duas categorias (Aprovado/Reprovado) através de uma terceira (Grupo STEM/Não-STEM).
    * **Construção do Gráfico:** Duas séries de barras são plotadas lado a lado para cada grupo no eixo X:
        * **Barras Verdes:** Representam o número de alunos **Aprovados**.
        * **Barras Vermelhas:** Representam o número de alunos **Reprovados**.
    * Para maior clareza, a função `ax.bar_label()` é utilizada para adicionar o valor numérico exato no topo de cada barra, facilitando a leitura e a interpretação precisa dos resultados.

&emsp; A análise deste gráfico permite comparar tanto os números absolutos de aprovados e reprovados entre os grupos, quanto a proporção de sucesso e insucesso dentro de cada grupo (comparando a altura da barra verde com a vermelha para STEM e não-STEM separadamente).

In [ ]:
sumario = df_onehot.groupby('STEM_SI')['Aprovou_Aprovou'].value_counts().unstack(fill_value=0)

labels = ['STEM_NO', 'STEM_SI']
aprovados = sumario[1].values
reprovados = sumario[0].values
x = np.arange(len(labels))
width = 0.35
fig, ax = plt.subplots(figsize=(8,6))

# Barras lado a lado
rects1 = ax.bar(x - width/2, aprovados, width, label='Aprovados', color='green')
rects2 = ax.bar(x + width/2, reprovados, width, label='Reprovados', color='red')

# Adicionar números em cima de cada barra
ax.bar_label(rects1, padding=3)  # padding é o espaço entre a barra e o número
ax.bar_label(rects2, padding=3)

# Labels e título
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Número de alunos')
ax.set_title('Aprovados e Reprovados por grupo STEM')
ax.legend()

plt.show()

#### Interpretação do Gráfico

&emsp; O gráfico de barras agrupado apresenta a contagem de alunos aprovados e reprovados, divididos entre os que pertencem a cursos de STEM (`STEM_SI`) e os que não pertencem (`STEM_NO`). A análise visual dos dados revela um insight importante sobre o perfil de desempenho dos alunos.

&emsp; Em termos absolutos, o número de reprovados no grupo não-STEM (48) é mais de quatro vezes maior que no grupo STEM (11). No entanto, essa comparação direta pode ser enganosa, uma vez que o número total de alunos no grupo não-STEM (911) é muito superior ao do grupo STEM (189).

&emsp; Para uma análise mais justa, é necessário comparar as taxas de reprovação proporcionais:
* **Taxa de Reprovação (Não-STEM):** `48 / (863 + 48) ≈ 5.3%`
* **Taxa de Reprovação (STEM):** `11 / (178 + 11) ≈ 5.8%`

&emsp; A análise proporcional mostra que as taxas de reprovação são, na verdade, muito semelhantes entre os dois grupos. Portanto, conclui-se que, para este conjunto de dados, **pertencer ou não a uma área de STEM não parece ser um fator determinante ou um forte preditor para a reprovação**, já que o desempenho relativo dos dois grupos é bastante parecido.

## 5.4. Análise Comparativa de Desempenho por Gênero

&emsp; De forma análoga à análise por grupo STEM, esta seção explora a relação entre o gênero dos alunos e seus resultados acadêmicos. O objetivo é criar uma visualização que permita comparar de forma clara as contagens de aprovação e reprovação entre os gêneros masculino e feminino, a fim de identificar possíveis disparidades no desempenho.

O processo para a criação do gráfico é o seguinte:

* **1. Agregação de Dados por Gênero:**
    * Os dados são agrupados pela variável `Genero_Masculino`. Dentro de cada um dos dois grupos (feminino e masculino), o método `.value_counts()` é aplicado para obter a contagem total de alunos aprovados e reprovados.
    * O resultado é reorganizado em uma tabela-resumo com o uso da função `.unstack()`, preparando os dados para a plotagem.

* **2. Preparação para a Visualização:**
    * As contagens numéricas de aprovados e reprovados para cada gênero são extraídas da tabela-resumo para alimentar o gráfico.

* **3. Geração do Gráfico de Barras Agrupado:**
    * **Justificativa da Escolha:** Novamente, o gráfico de barras agrupado é a ferramenta visual mais adequada, pois permite uma comparação direta e intuitiva das contagens de resultado (Aprovado/Reprovado) para cada categoria de gênero.
    * **Construção do Gráfico:** O gráfico é construído com duas séries de barras lado a lado:
        * **Barras Verdes:** Representam o número de alunos **Aprovados**.
        * **Barras Vermelhas:** Representam o número de alunos **Reprovados**.
    * Para facilitar a leitura precisa, os valores exatos são adicionados no topo de cada barra.

&emsp; Esta visualização permite analisar se existem diferenças notáveis tanto no número absoluto de reprovações quanto na proporção de insucesso (tamanho da barra vermelha em relação à verde) entre os gêneros neste conjunto de dados.

In [ ]:

sumario = df_onehot.groupby('Genero_Masculino')['Aprovou_Aprovou'].value_counts().unstack(fill_value=0)

labels = ['Genero_Femenino', 'Genero_Masculino']
aprovados = sumario[1].values
reprovados = sumario[0].values
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8,6))

# Barras lado a lado
rects1 = ax.bar(x - width/2, aprovados, width, label='Aprovados', color='green')
rects2 = ax.bar(x + width/2, reprovados, width, label='Reprovados', color='red')

# Adicionar números em cima de cada barra
ax.bar_label(rects1, padding=3)  # padding é o espaço entre a barra e o número
ax.bar_label(rects2, padding=3)

# Labels e título
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Número de alunos')
ax.set_title('Aprovados e Reprovados por gênero')
ax.legend()

plt.show()


#### Interpretação do Gráfico

&emsp; Este gráfico compara as contagens absolutas de alunos aprovados e reprovados entre os gêneros feminino e masculino. A visualização mostra que, embora os números totais de alunos em cada grupo sejam semelhantes, existem diferenças nas taxas de reprovação que merecem ser analisadas.

&emsp; Para uma comparação precisa, é necessário calcular a taxa de reprovação proporcional para cada grupo:
* **Taxa de Reprovação (Gênero Feminino):** `26 / (533 + 26) ≈ 4.7%`
* **Taxa de Reprovação (Gênero Masculino):** `33 / (508 + 33) ≈ 6.1%`

&emsp; A análise proporcional indica que, neste conjunto de dados, a taxa de reprovação do grupo masculino é ligeiramente superior à do grupo feminino. Embora a diferença não seja expressiva, ela sugere que o gênero pode ser uma variável com uma leve influência preditiva no desempenho dos alunos, o que ajuda a contextualizar por que esta *feature* frequentemente aparece como relevante nas análises de interpretabilidade dos modelos (SHAP).

## 5.5. Nota média do quizz com média de tempo gasto do dataframe do período 1

&emsp; Para entender a progressão do desempenho dos alunos ao longo do tempo, esta análise visualiza a evolução de duas métricas chave: as **notas médias** e os **tempos médios de conclusão** para os seis primeiros quizzes. O objetivo é identificar tendências e possíveis correlações entre o conhecimento adquirido (refletido na nota) e a eficiência (refletida no tempo).

O processo para a criação do gráfico é o seguinte:

* **1. Cálculo das Médias por Quiz:**
    * Primeiramente, são calculadas as médias de notas e de tempos para cada um dos quizzes de 1 a 6. O resultado são duas listas de valores que representam a performance média da turma em cada avaliação.

* **2. Geração do Gráfico de Eixo Duplo (*Dual-Axis Chart*):**
    * **Justificativa da Escolha:** Como as notas (escala de 0 a 10) e os tempos (escala em segundos, ex: 0 a 300) possuem grandezas muito diferentes, plotá-los em um mesmo eixo Y tornaria uma das linhas praticamente ilegível. A solução é um gráfico de eixo duplo, criado com a função `ax.twinx()`, que permite comparar as duas tendências de forma clara, cada uma com sua própria escala.
    * **Construção do Gráfico:**
        * O **eixo X** representa a sequência dos seis quizzes.
        * A **linha azul**, correspondente ao **eixo Y esquerdo**, plota a evolução das **Notas Médias**.
        * A **linha vermelha**, correspondente ao **eixo Y direito**, plota a evolução do **Tempo Médio** de conclusão.

&emsp; A análise deste gráfico permite observar se há uma curva de aprendizado (notas aumentando), se os alunos se tornam mais rápidos com o tempo (tempos diminuindo) e, principalmente, como essas duas tendências se relacionam. Por exemplo, uma queda no tempo acompanhada de um aumento nas notas pode sugerir um ganho de eficiência e domínio do conteúdo.

In [ ]:
# Calcular médias por quiz
notas_medias = [df1_analise[f"Quiz{i}"].mean() for i in range(1, 7)]
tempos_medios = [df1_analise[f"TempoQ{i}"].mean() for i in range(1, 7)]

quizzes = range(1, 7)

fig, ax1 = plt.subplots(figsize=(8,5))

# Notas
ax1.plot(quizzes, notas_medias, marker='o', color='blue', label='Notas médias (0-10)')
ax1.set_xlabel("Quizzes (Q1–Q6)")
ax1.set_ylabel("Notas médias", color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_xticks(quizzes)

# Tempos
ax2 = ax1.twinx()
ax2.plot(quizzes, tempos_medios, marker='s', color='red', label='Tempo médio (s)')
ax2.set_ylabel("Tempo médio (segundos)", color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title("Evolução de notas e tempos de execução nos quizzes")
fig.tight_layout()
plt.show()


#### Interpretação do Gráfico

&emsp; O gráfico de eixo duplo permite a análise simultânea da evolução das notas médias (linha azul) e dos tempos médios de conclusão (linha vermelha) ao longo dos seis primeiros quizzes. A visualização dos dados do primeiro semestre revela uma relação clara e consistente entre as duas métricas.

&emsp; A principal observação é a **forte relação inversa** entre o desempenho (notas) e o esforço (tempo). Geralmente, quando a média de notas de um quiz diminui, o tempo médio para completá-lo aumenta, e vice-versa. Este padrão é particularmente evidente ao comparar o **Quiz 4** com o **Quiz 5**: o Quiz 4 registrou a nota média mais baixa do período e, simultaneamente, o maior tempo médio de execução. Em contraste, no Quiz 5, a nota média atingiu seu pico enquanto o tempo de conclusão diminuiu consideravelmente em relação ao quiz anterior.

&emsp; Esta análise sugere que o tempo de execução pode ser interpretado como um forte indicador da dificuldade percebida das avaliações. Quizzes onde os alunos, em média, levaram mais tempo foram também aqueles com as menores notas, indicando uma maior dificuldade no conteúdo. Por outro lado, um bom desempenho parece estar associado a uma maior eficiência na resolução das questões.

## 5.6. Nota média do quizz com média de tempo gasto do dataframe do período 2

&emsp; Repetindo a mesma análise para os dados do segundo semestre (`df2_analise`), esta seção visa traçar a curva de aprendizado e eficiência dos alunos neste período. O objetivo é visualizar as tendências das notas e tempos médios e, crucialmente, comparar esses padrões com os observados no primeiro semestre.

O processo para a criação do gráfico é idêntico ao anterior:

* **1. Cálculo das Médias por Quiz:**
    * São calculadas as médias de notas e tempos para cada um dos quizzes de 1 a 6, desta vez utilizando os dados do segundo semestre.

* **2. Geração do Gráfico de Eixo Duplo (*Dual-Axis Chart*):**
    * **Justificativa da Escolha:** A utilização de um gráfico de eixo duplo continua sendo a abordagem ideal para comparar as duas métricas de escalas distintas (notas e segundos) em uma única visualização.
    * **Construção do Gráfico:** O gráfico é configurado da mesma forma, com o **eixo Y esquerdo (azul)** representando as **Notas Médias** e o **eixo Y direito (vermelho)** representando o **Tempo Médio**.

&emsp; A principal análise deste gráfico consiste em comparar as tendências observadas aqui com as do gráfico anterior. Deve-se observar se a curva de aprendizado (tendência das notas) e de eficiência (tendência dos tempos) foi semelhante, melhor ou pior no segundo semestre, buscando identificar diferenças ou semelhanças no comportamento das turmas.

In [ ]:
# Calcular médias por quiz
notas_medias = [df2_analise[f"Quiz{i}"].mean() for i in range(1, 7)]
tempos_medios = [df2_analise[f"TempoQ{i}"].mean() for i in range(1, 7)]

quizzes = range(1, 7)

fig, ax1 = plt.subplots(figsize=(8,5))

# Notas
ax1.plot(quizzes, notas_medias, marker='o', color='blue', label='Notas médias (0-10)')
ax1.set_xlabel("Quizzes (Q1–Q6)")
ax1.set_ylabel("Notas médias", color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_xticks(quizzes)

# Tempos
ax2 = ax1.twinx()
ax2.plot(quizzes, tempos_medios, marker='s', color='red', label='Tempo médio (s)')
ax2.set_ylabel("Tempo médio (segundos)", color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title("Evolução de notas e tempos de execução nos quizzes")
fig.tight_layout()
plt.show()


#### Interpretação do Gráfico

&emsp; O gráfico de eixo duplo permite a análise simultânea da evolução das notas médias (linha azul) e dos tempos médios de conclusão (linha vermelha) ao longo dos seis primeiros quizzes. A visualização dos dados do primeiro semestre revela uma relação clara e consistente entre as duas métricas.

&emsp; A principal observação é a **forte relação inversa** entre o desempenho (notas) e o esforço (tempo). Geralmente, quando a média de notas de um quiz diminui, o tempo médio para completá-lo aumenta, e vice-versa. Este padrão é particularmente evidente ao comparar o **Quiz 4** com o **Quiz 5**: o Quiz 4 registrou a nota média mais baixa do período e, simultaneamente, o maior tempo médio de execução. Em contraste, no Quiz 5, a nota média atingiu seu pico enquanto o tempo de conclusão diminuiu consideravelmente em relação ao quiz anterior.

&emsp; Esta análise sugere que o tempo de execução pode ser interpretado como um forte indicador da dificuldade percebida das avaliações. Quizzes onde os alunos, em média, levaram mais tempo foram também aqueles com as menores notas, indicando uma maior dificuldade no conteúdo. Por outro lado, um bom desempenho parece estar associado a uma maior eficiência na resolução das questões.

## 5.7. Análise de Estatísticas Descritivas

&emsp; Após as análises visuais, esta seção realiza um aprofundamento quantitativo nas principais variáveis de desempenho do `df_consolidado`. O objetivo é gerar uma tabela-resumo com as principais medidas estatísticas, como média, desvio padrão e quartis. Esta análise serve tanto como uma validação final da integridade dos dados após o processamento, quanto para fornecer um panorama geral sobre a distribuição e a escala de cada variável.

O processo é executado em duas etapas:

* **1. Seleção das Variáveis de Interesse:**
    * Primeiramente, é definida uma lista contendo as colunas de maior relevância para a análise de desempenho, incluindo notas de quizzes, tempos de resolução, projetos e avaliações parciais. Esta seleção focada permite gerar um resumo conciso e direcionado.

* **2. Geração da Tabela Descritiva:**
    * O método `.describe()` da biblioteca Pandas é aplicado ao subconjunto de dados selecionado. Esta função calcula automaticamente as seguintes estatísticas para cada coluna:
        * **`count`**: Número de observações não nulas.
        * **`mean`**: A média dos valores.
        * **`std`**: O desvio padrão, que indica a dispersão dos dados em torno da média.
        * **`min` e `max`**: Os valores mínimo e máximo, úteis para verificar o intervalo de cada variável.
        * **`25%`, `50%`, `75%`**: Os quartis, que dividem os dados em quatro partes iguais e ajudam a entender a distribuição.
    * Ao final, a tabela é transposta (`.T`) para que as variáveis fiquem nas linhas, um formato que facilita a leitura e a comparação.

In [ ]:
# seleciona todas as colunas de interesse
colunas = [
    "Quiz1", "TempoQ1", "Quiz2", "TempoQ2", "Quiz3", "TempoQ3", 
    "Quiz4", "TempoQ4", "Quiz5", "TempoQ5", "Quiz6", "TempoQ6", 
    "Quiz7", "TempoQ7", "Quiz8", "Nota_Oficial", 
    "Projeto_Parte1", "Projeto_Parte2", "Quanto_Melhora", "Oficinas", "Parcial_1",
    "Parcial_2"
]       

# gera tabela descritiva
desc_tabela = df_consolidado[colunas].describe().T  # Transpõe para variáveis ficarem como linhas


## 5.8. Análise da Distribuição das Notas por Avaliação

&emsp; Após a análise das estatísticas descritivas, esta seção aprofunda a investigação sobre o desempenho dos alunos através de uma análise visual. O objetivo é gerar uma série de **histogramas** para visualizar a **distribuição das notas** em cada uma das principais avaliações, incluindo quizzes, projetos e a nota oficial. Esta abordagem permite entender a forma, a tendência central e a dispersão das notas, revelando padrões que não são aparentes em tabelas de resumo.

O processo de criação dos gráficos segue as etapas abaixo:

* **1. Seleção das Variáveis e Estrutura do Gráfico:**
    * Primeiramente, é definida uma lista com todas as colunas de avaliação de interesse.
    * Em seguida, uma grade de subplots (4x3) é criada com a função `plt.subplots`. Esta estrutura permite que todos os histogramas sejam exibidos em uma única figura, facilitando a comparação direta entre eles.

* **2. Geração Iterativa dos Histogramas:**
    * O código itera sobre cada variável de avaliação para plotar um histograma individual em um dos subplots.
    * O histograma (`ax.hist`) agrupa os alunos em faixas de notas (`bins`), exibindo a contagem de alunos em cada faixa. Isso revela a concentração de notas e a forma geral da distribuição para cada atividade.

* **3. Adição de Linhas de Tendência Central:**
    * Para cada histograma, duas linhas verticais são sobrepostas para marcar as medidas de tendência central:
        * **Média (linha azul tracejada):** Representa a nota média da turma.
        * **Mediana (linha vermelha contínua):** Representa a nota central, que divide a turma exatamente ao meio.
    * **Justificativa:** A comparação visual entre a média e a mediana é uma forma eficaz de avaliar a **assimetria (skewness)** da distribuição. Se a média é significativamente diferente da mediana, isso sugere que a distribuição é assimétrica, provavelmente devido a um conjunto de notas muito altas ou muito baixas que "puxam" a média em sua direção.

* **4. Padronização e Finalização:**
    * Cada subplot é configurado com títulos, rótulos e uma legenda para garantir a clareza. A função `plt.tight_layout()` é usada para ajustar o espaçamento e evitar a sobreposição de elementos, resultando em uma figura final limpa e profissional.

&emsp; A análise desta grade de gráficos permite comparar a dificuldade percebida e a distribuição de desempenho entre as diferentes avaliações. Por exemplo, distribuições com "cauda" à esquerda (média menor que a mediana) geralmente indicam avaliações mais fáceis para a maioria, enquanto distribuições com "cauda" à direita (média maior que a mediana) podem indicar o contrário.

In [ ]:
# Lista de quizzes + projetos + nota oficial
avaliacoes = [f"Quiz{i}" for i in range(1, 9)] + ["Projeto_Parte1", "Projeto_Parte2", "Nota_Oficial", "Oficinas"]

# Definir bins de 0 até 5 com passo 0.25
bins = np.arange(0, 5.25, 0.5)  # inclui 5

# Criar grade de subplots (4x3 para 11 avaliações)
fig, axes = plt.subplots(4, 3, figsize=(16, 14))
axes = axes.flatten()

for i, avaliacao in enumerate(avaliacoes):
    ax = axes[i]
    
    # Histograma com bins fixos
    ax.hist(df_consolidado[avaliacao], bins=bins, color="skyblue", edgecolor="black", rwidth=0.9)
    
    # Calcular média e mediana
    media = df_consolidado[avaliacao].mean()
    mediana = df_consolidado[avaliacao].median()
    
    # Linhas de média e mediana
    ax.axvline(media, color="blue", linestyle="--", linewidth=2, label=f"Média: {media:.2f}")
    ax.axvline(mediana, color="red", linestyle="-", linewidth=2, label=f"Mediana: {mediana:.2f}")
    
    # Configurações do gráfico
    ax.set_title(f"Distribuição - {avaliacao}")
    ax.set_xlabel("Nota")
    ax.set_ylabel("Quantidade de Alunos")
    ax.set_xticks(bins)  # força ticks de 0 a 5 de 0.25 em 0.25
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.legend()

# Remove eixos extras se sobrar espaços (caso 12 slots, mas só 11 avaliações)
for j in range(len(avaliacoes), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Distribuição de Notas (Quizzes, Projetos e Nota Oficial)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## 5.9. Teste de Normalidade para a Média dos Quizzes

&emsp; Após a análise visual das distribuições através de histogramas, esta seção aplica um teste estatístico formal para verificar se a variável `CalcNotaQuiz` (que representa a média das notas dos quizzes de cada aluno) segue uma distribuição normal (Gaussiana). A verificação da normalidade é um passo importante em análises estatísticas, pois muitos métodos paramétricos assumem que os dados seguem esta distribuição.

O processo é realizado da seguinte forma:

* **1. Seleção e Preparação dos Dados:**
    * A variável de interesse, `CalcNotaQuiz`, é selecionada do dataframe `df_consolidado`.
    * Quaisquer valores ausentes (`NaN`) são removidos com o método `.dropna()`, pois o teste estatístico não pode ser executado com dados faltantes.

* **2. Aplicação do Teste de Shapiro-Wilk:**
    * É utilizado o teste de Shapiro-Wilk, implementado na função `shapiro` da biblioteca `SciPy`. Este é um dos testes mais poderosos e recomendados para avaliar a normalidade de uma amostra de dados.

* **3. Definição das Hipóteses Estatísticas:**
    * O teste de Shapiro-Wilk opera com base em duas hipóteses concorrentes:
        * **Hipótese Nula (H₀):** A amostra de dados provém de uma população com distribuição normal.
        * **Hipótese Alternativa (H₁):** A amostra de dados *não* provém de uma população com distribuição normal.

* **4. Interpretação do Resultado com base no p-valor:**
    * O resultado do teste é interpretado através do **p-valor**, utilizando um nível de significância padrão de 5% (α = 0.05). A regra de decisão é:
        * Se o **p-valor for maior que 0.05**, não há evidência estatística suficiente para descartar a normalidade. Portanto, **não se rejeita a Hipótese Nula (H₀)**.
        * Se o **p-valor for menor ou igual a 0.05**, a probabilidade de os dados virem de uma distribuição normal é muito baixa. Portanto, **rejeita-se a Hipótese Nula (H₀)** em favor da Hipótese Alternativa (H₁).

&emsp; A conclusão impressa pela célula de código fornece um veredito estatístico sobre a distribuição da variável `CalcNotaQuiz`, informando se ela pode ou não ser considerada normal para fins de análises estatísticas subsequentes.

In [ ]:
# Testes de normalidade das variáveis "CalcNotaQuiz", "Talleres", "Calificación_Oficial"
# Testando normalidade para "CalcNotaQuiz" via Shapiro-Wilk
from scipy.stats import shapiro

dados = df_consolidado['CalcNotaQuiz'].dropna()

stat, p = shapiro(dados)

print('H0: Os dados seguem uma distribuição normal')
print('H1: Os dados não seguem uma distribuição normal')
print("Estatística W:", stat)
print("p-valor:", p)

if p > 0.05:
    print("Não rejeitamos H0: os dados seguem uma distribuição normal.")
else:
    print("Rejeitamos H0: os dados não seguem uma distribuição normal.")

#### Interpretação do Resultado

&emsp; Conforme o resultado do teste de Shapiro-Wilk apresentado acima, o **p-valor** obtido foi de `4.78e-24`, um valor extremamente próximo de zero.

&emsp; Sendo este valor muito inferior ao nível de significância padrão de 0.05 (α = 0.05), a Hipótese Nula (H₀) de que os dados seguem uma distribuição normal é **fortemente rejeitada**.

&emsp; Portanto, conclui-se com alta confiança estatística que a distribuição da variável `CalcNotaQuiz` não é normal (Gaussiana). Este achado é importante pois confirma que métodos estatísticos que dependem da premissa de normalidade podem não ser os mais adequados para analisar esta variável em profundidade.

## 5.10. Análise Visual da Distribuição das Oficinas

&emsp; Antes de realizar o teste estatístico formal de normalidade para a variável `Oficinas`, esta seção realiza uma análise visual aprofundada de sua distribuição. O objetivo é criar uma visualização composta que ofereça múltiplos insights sobre as notas obtidas pelos alunos nesta atividade.

&emsp; Para isso, é gerado um gráfico com dois subplots utilizando a biblioteca `Plotly`:

* **Histograma:** Este gráfico exibe a **frequência** de alunos por faixa de nota. Ele é útil para identificar rapidamente os picos de concentração e a forma geral da distribuição.
* **Violin Plot:** Este gráfico complementa o histograma ao mostrar a **densidade de probabilidade** dos dados. A forma do "violino" ilustra onde as notas são mais ou menos comuns. Adicionalmente, ele contém um *box plot* interno, que resume a mediana e os quartis, e uma linha que marca a média, permitindo uma análise detalhada da tendência central e da dispersão.

&emsp; A combinação desses dois gráficos em uma única figura fornece uma compreensão visual completa da variável `Oficinas`.

In [ ]:
# Criar subplots com 2 linhas e 1 coluna para acomodar o histograma e o violin plot
fig = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=('Histograma de Oficinas', 'Violin Plot de Oficinas'), # Títulos para cada subplot
    vertical_spacing=0.15  # Aumentar o espaço vertical entre os subplots para evitar sobreposição de títulos/rótulos
)

# Adicionar o histograma para a coluna 'Oficinas'
fig.add_trace(
    go.Histogram(
        x=df_consolidado['Oficinas'], # Dados para o histograma
        nbinsx=30, # Número de bins no histograma
        name='', # Nome da trace (não exibido com showlegend=False)
        marker=dict(
            color='#004066', # Cor das barras do histograma
            opacity=0.8, # Opacidade das barras
            line=dict(color='black', width=1) # Adiciona borda preta às barras
        )
    ),
    row=1, col=1 # Posicionar no primeiro subplot (primeira linha, primeira coluna)
)

# Adicionar o violin plot para a coluna 'Oficinas'
fig.add_trace(
    go.Violin(
        y=df_consolidado['Oficinas'], # Dados para o violin plot
        name='', # Nome da trace
        marker_color='#004066', # Cor do violin plot
        opacity=0.8, # Opacidade
        box_visible=True, # Mostrar box plot dentro do violin plot
        meanline_visible=True # Mostrar linha da média
    ),
    row=2, col=1 # Posicionar no segundo subplot (segunda linha, primeira coluna)
)


# Atualizar configurações gerais de layout do gráfico
fig.update_layout(
    title_text="Análise de Oficinas - Multiplot", # Título principal do gráfico
    height=900, # Altura do gráfico em pixels
    showlegend=False, # Não mostrar a legenda
    plot_bgcolor='white', # Cor de fundo da área de plotagem
    paper_bgcolor='white', # Cor de fundo do papel (área fora da plotagem)
    font=dict(family="Arial"), # Fonte do texto do gráfico
    margin=dict(t=80) # Aumentar margem superior para o título principal não ficar muito próximo do topo
)

# Atualizar layout dos eixos do histograma
fig.update_xaxes(title_text="Nota de Oficinas", row=1, col=1, gridcolor='lightgray', zerolinecolor='lightgray') # Rótulo do eixo X
fig.update_yaxes(title_text="Frequência", row=1, col=1, gridcolor='lightgray', zerolinecolor='lightgray') # Rótulo do eixo Y

# Atualizar layout dos eixos do violin plot
fig.update_xaxes(title_text="", row=2, col=1, gridcolor='lightgray', zerolinecolor='lightgray') # Remover rótulo do eixo X para evitar redundância
fig.update_yaxes(title_text="Nota de Oficinas", row=2, col=1, gridcolor='lightgray', zerolinecolor='lightgray') # Rótulo do eixo Y


fig.update_layout(height=700, width=1000) # Ajustar altura e largura finais do gráfico
# Mostrar o gráfico
print(df_consolidado['Oficinas'].dtype)
print(df_consolidado['Oficinas'].head())


fig.show(renderer="notebook")



#### Interpretação do Gráfico

&emsp; A análise visual da variável `Oficinas` revela uma distribuição de notas bastante particular e informativa. A observação dos dois gráficos leva às seguintes conclusões:

* **Distribuição Bimodal e Assimétrica:** A característica mais marcante é a natureza **bimodal** da distribuição, ou seja, a presença de dois picos distintos. Há um pico massivo e dominante na nota máxima (5.0), indicando que a grande maioria dos alunos obteve um desempenho excelente nas oficinas. Um segundo pico, muito menor, é visível na extremidade inferior da escala (em torno de 1.5), representando um pequeno grupo de alunos com desempenho muito baixo.

* **Assimetria à Esquerda:** A concentração de notas altas torna a distribuição fortemente **assimétrica à esquerda (*left-skewed*)**. Isso é confirmado no *violin plot*, onde a média (linha tracejada) está visivelmente abaixo da mediana (linha contínua), sendo "puxada" para baixo pelo grupo de notas mais baixas.

&emsp; Em resumo, a atividade de oficinas parece ter polarizado os alunos em dois grupos: uma vasta maioria que a concluiu com sucesso máximo e uma minoria que encontrou dificuldades significativas. Essa distribuição distinta justifica o resultado do teste de Shapiro-Wilk, que rejeitará a hipótese de normalidade.

## 5.11. Teste de Normalidade para a Nota das Oficinas

&emsp; Seguindo a mesma metodologia da seção anterior, esta análise aplica um teste estatístico formal para verificar se a variável `Oficinas` segue uma distribuição normal (Gaussiana). O objetivo é determinar a natureza da distribuição desta variável para orientar a escolha de métodos estatísticos apropriados em análises futuras.

O processo executado pelo código é o seguinte:

* **1. Seleção e Preparação dos Dados:**
    * A variável de interesse, `Oficinas`, é selecionada do dataframe `df_consolidado`.
    * Quaisquer valores ausentes (`NaN`) são removidos com o método `.dropna()` para garantir a integridade do conjunto de dados que será testado.

* **2. Aplicação do Teste de Shapiro-Wilk:**
    * É utilizado o teste de Shapiro-Wilk, implementado na função `shapiro` da biblioteca `SciPy`. Este teste é especificamente desenhado para avaliar a hipótese de normalidade em uma amostra de dados.

* **3. Definição das Hipóteses Estatísticas:**
    * O teste opera com base em duas hipóteses concorrentes:
        * **Hipótese Nula (H₀):** A amostra de dados provém de uma população com distribuição normal.
        * **Hipótese Alternativa (H₁):** A amostra de dados *não* provém de uma população com distribuição normal.

* **4. Interpretação do Resultado com base no p-valor:**
    * O resultado do teste é interpretado através do **p-valor**, utilizando um nível de significância padrão de 5% (α = 0.05). A regra de decisão é:
        * Se o **p-valor for maior que 0.05**, não há evidência estatística suficiente para descartar a normalidade. Portanto, **não se rejeita a Hipótese Nula (H₀)**.
        * Se o **p-valor for menor ou igual a 0.05**, a probabilidade de os dados virem de uma distribuição normal é muito baixa. Portanto, **rejeita-se a Hipótese Nula (H₀)**.

&emsp; A saída da célula de código apresenta a conclusão estatística sobre a normalidade da variável `Oficinas`, um resultado importante para a validação das premissas de futuras análises estatísticas.

In [ ]:
# Testando normalidade para "Oficinas" (Talleres) via Shapiro-Wilk
dados = df_consolidado['Oficinas'].dropna()

stat, p = shapiro(dados)

print('H0: Os dados seguem uma distribuição normal')
print('H1: Os dados não seguem uma distribuição normal')
print("Estatística W:", stat)
print("p-valor:", p)

if p > 0.05:
    print("Não rejeitamos H0: os dados seguem uma distribuição normal.")
else:
    print("Rejeitamos H0: os dados não seguem uma distribuição normal.")

#### Interpretação do Resultado

&emsp; De forma análoga ao teste anterior, o resultado para a variável `Oficinas` apresenta um **p-valor** de `1.11e-35`, um valor efetivamente zero e muito inferior ao nível de significância de 0.05.

&emsp; Com base neste resultado, a Hipótese Nula (H₀) de que os dados seguem uma distribuição normal é **novamente rejeitada** com um altíssimo grau de confiança estatística.

&emsp; A análise confirma, portanto, que a distribuição das notas das `Oficinas` também não é normal. A consistência deste achado entre diferentes variáveis de desempenho sugere que distribuições assimétricas são uma característica comum neste conjunto de dados.

## 5.12. Teste de Normalidade para a Nota Oficial

&emsp; Para concluir a verificação estatística das distribuições, esta seção aplica o teste de normalidade de Shapiro-Wilk à variável `Nota_Oficial`. O objetivo é determinar formalmente se a distribuição das notas finais dos alunos se aproxima de uma curva normal, um passo importante para validar as premissas de certos métodos estatísticos.

O processo executado pelo código é o seguinte:

* **1. Seleção e Preparação dos Dados:**
    * A variável `Nota_Oficial` é selecionada do dataframe `df_consolidado` e quaisquer valores ausentes são removidos.

* **2. Aplicação do Teste de Shapiro-Wilk:**
    * A função `shapiro` da biblioteca `SciPy` é utilizada para testar a hipótese de normalidade da amostra.

* **3. Definição das Hipóteses e Interpretação:**
    * O teste avalia a **Hipótese Nula (H₀)** de que os dados seguem uma distribuição normal. Com base no **p-valor** resultante e um nível de significância de 0.05, o código determinará se há evidência suficiente para rejeitar ou não esta hipótese.

In [ ]:
# Testando normalidade para "Nota_Oficial" (Calificación_Oficial) via Shapiro-Wilk
from scipy.stats import shapiro

dados = df_consolidado['Nota_Oficial'].dropna()

stat, p = shapiro(dados)

print('H0: Os dados seguem uma distribuição normal')
print('H1: Os dados não seguem uma distribuição normal')
print("Estatística W:", stat)
print("p-valor:", p)


if p > 0.05:
    print("Não rejeitamos H0: os dados seguem uma distribuição normal.")
else:
    print("Rejeitamos H0: os dados não seguem uma distribuição normal.")

#### Interpretação do Resultado

&emsp; O resultado do teste de Shapiro-Wilk para a variável `Nota_Oficial` apresentou um **p-valor** de `4.03e-28`, um valor extremamente baixo e muito inferior ao nível de significância de 0.05.

&emsp; Com base neste resultado, a Hipótese Nula (H₀) de que os dados seguem uma distribuição normal é **fortemente rejeitada**.

&emsp; Esta análise finaliza a verificação de normalidade, e o resultado é consistente com os testes anteriores: as principais variáveis de desempenho contínuo neste conjunto de dados (`CalcNotaQuiz`, `Oficinas` e `Nota_Oficial`) não seguem uma distribuição normal.

# 3. Treino e teste do modelo Definitivo: Nearest Centroid

&emsp; O Nearest Centroid é um algoritmo de aprendizado supervisionado conhecido por sua alta eficiência, simplicidade e interpretabilidade. Diferente de modelos complexos baseados em árvores, sua lógica é fundamentalmente geométrica. O princípio do algoritmo é calcular um ponto central para cada classe, conhecido como **centroide**, que representa a média de todas as amostras de treinamento daquela classe. Matematicamente, o centroide $C_k$ para a classe $k$ é a média vetorial de todas as amostras $x_i$ pertencentes a ela:

$$
C_k = \frac{1}{|N_k|} \sum_{i \in N_k} x_i
$$

&emsp; Uma vez que os centroides são definidos, a classificação de um novo dado ocorre de forma intuitiva: o novo ponto é atribuído à classe do centroide do qual ele está mais próximo. A "proximidade" é calculada usando uma métrica de distância, sendo a **distância Euclidiana** (a distância em linha reta entre dois pontos) a mais comum. A principal vantagem deste método é a transparência, pois os próprios centroides podem ser inspecionados para entender o "perfil médio" de cada classe.

&emsp; No contexto deste projeto, o Nearest Centroid foi escolhido como modelo preditivo final para prever se os alunos irão reprovar. O algoritmo calcula dois perfis centrais: o "aluno reprovado médio" e o "aluno aprovado médio". A previsão para cada estudante é então determinada pela sua proximidade a um desses dois perfis, com base as notas nos quizzes, tempo gasto fazendo eles, nota da primeira parcial, se é do gênero masculino e se é de curso de tecnologia.

#### O Tratamento de Dados Desbalanceados com SMOTE

&emsp; Um dos principais desafios em problemas de classificação como este é o **desbalanceamento de classes**. Em nosso conjunto de dados, o número de alunos aprovados, a grande maioria, é muito maior que o de reprovados, a minoria e o foco do modelo. Se um modelo for treinado diretamente com esses dados, ele pode se tornar enviesado, aprendendo a prever a classe dominante e ignorando a classe de interesse, resultando em predições ineficazes para identificar os alunos em risco de reprovação.

&emsp; Para mitigar esse problema, utilizamos a técnica **SMOTE (Synthetic Minority Over-sampling Technique)**. Diferente de uma simples duplicação dos dados da classe minoritária, o SMOTE gera **novos exemplos sintéticos** que são plausíveis e representativos dos alunos reprovados. O algoritmo funciona da seguinte forma:
1.  Seleciona um aluno da classe minoritária, dos reprovados.
2.  Identifica seus "vizinhos" mais próximos, outros alunos que foram reprovados.
3.  Cria um novo dado sintético em um ponto aleatório na linha que conecta o aluno original a um de seus vizinhos.

&emsp; Essa abordagem cria um conjunto de dados de treino mais rico e balanceado, forçando o modelo a aprender as características que definem um aluno reprovado com mais atenção. É fundamental destacar que o SMOTE foi aplicado **exclusivamente no conjunto de dados de treino**, após a separação do conjunto de teste. Essa prática é crucial para evitar o **vazamento de dados (data leakage)** e garantir que a performance do modelo seja realista e fidedigna.

#### Explicação dos Parâmetros Usados

&emsp; Embora o Nearest Centroid seja um modelo simples, a otimização de seus hiperparâmetros via `GridSearchCV` é uma etapa importante para maximizar sua performance. A busca focou em encontrar a melhor combinação de métrica de distância e regularização para o nosso problema.

- **metric**: Define a fórmula matemática usada para calcular a distância entre um ponto e os centroides. As opções testadas foram:
    - `'euclidean'`: A distância de linha reta padrão. É sensível à escala das variáveis.
    - `'manhattan'`: A "distância de quarteirão", calculada como a soma das diferenças absolutas entre as coordenadas. É frequentemente mais robusta a outliers.
- **shrink_threshold**: Atua como um parâmetro de regularização, que pode ajudar a melhorar a generalização do modelo. Quando um valor é definido, ele "encolhe" os centroides de cada classe em direção ao centroide geral de todos os dados. Isso diminui o impacto de ruídos ou outliers em uma classe específica, evitando que o centroide daquela classe fique muito distante. Foram testados valores de `0.1`, `0.5` e `1.0` para encontrar o nível ótimo de "encolhimento".
- **GridSearchCV(cv=5)**: A validação cruzada em 5 folds foi utilizada para garantir que a performance de cada combinação de parâmetros fosse estável e robusta, reduzindo o risco de uma escolha enviesada.
- **scoring = 'recall'**: O **recall** foi definido como a métrica principal de otimização. Esta escolha reforça o objetivo de negócio do projeto: priorizar a identificação do maior número possível de alunos em risco de reprovação (minimizar os falsos negativos).

In [ ]:
def nearest_centroid(x_train, x_test, y_train, y_test):
    # x_train, x_test, y_train, y_test = train_test_split(X1, y1,test_size=0.2, random_state=42)

    sm = SMOTE(random_state=42)
    x_train, y_train = sm.fit_resample(x_train, y_train)

    # Definição do classificador
    nc = NearestCentroid()

    # Grid de parâmetros
    param_grid = {
        "metric": ["euclidean", "manhattan"],  
        "shrink_threshold": [None, 0.1, 0.5, 1.0]
    }

    # Aplicação do GridSearchCV
    grid = GridSearchCV(nc, param_grid, cv=5, scoring='recall')
    grid.fit(x_train, y_train)
    melhor_modelo = grid.best_estimator_
    print("Melhores parâmetros:", melhor_modelo)
    y_pred = grid.predict(x_test)

    # Métricas obtidas
    print("\n==== Métricas obtidas ====")
    recall = recall_score(y_test, y_pred, pos_label=1)
    print("Recall da classe 1 no teste: ", recall)
    f1score = f1_score(y_test, y_pred, pos_label=1)
    print("F1 score da classe 1 no teste: ", f1score)
    confusion_matrix(y_test, y_pred, labels=[1,0])
    disp = ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        display_labels=["Reprovado (1)", "Aprovado (0)"],  
        cmap="Blues",
        values_format="d",
        labels = [1,0]
    )

    plt.title("Matriz de Confusão - NearestCentroid")
    plt.show()

    # Interpretabilidade com SHAP
    explainer = shap.KernelExplainer(melhor_modelo.predict, x_train)
    shap_values = explainer.shap_values(x_test)
    print("Gerando gráfico SHAP...")
    shap.summary_plot(shap_values, x_test)

### 3.1 Análise Preditiva Precoce (Semana 6)

&emsp; Iniciando a aplicação do modelo definitivo, esta seção detalha a performance do **Nearest Centroid** na previsão da Semana 6. O objetivo é avaliar a eficácia do modelo como um sistema de **alerta precoce**, utilizando as notas até o quiz 3.

&emsp; Além de avaliar a performance preditiva através da Matriz de Confusão, Recall e F1-Score, esta análise também inclui a interpretabilidade dos resultados através do **SHAP (SHapley Additive exPlanations)**. Esta técnica nos permitirá entender exatamente quais features mais influenciaram as previsões do modelo, adicionando uma camada de transparência à solução.

In [ ]:
nearest_centroid(X1_train, X1_test, y1_train, y1_test)

#### Análise dos Resultados (Semana 6)

&emsp; Na otimização do modelo para a Semana 6, o `GridSearchCV` determinou que a métrica de distância **'manhattan'**, sem o uso de regularização (`shrink_threshold`), apresentou o melhor desempenho para a tarefa de alerta precoce.

&emsp; As métricas de avaliação, focadas na identificação da classe de reprovados (classe 1), foram as seguintes:

- **Recall da classe 1 (Reprovado):** 0.808 (80.8%)
- **F1-Score da classe 1 (Reprovado):** 0.429 (42.9%)

**Interpretabilidade do Modelo com SHAP**

&emsp; O gráfico SHAP revela os fatores que mais impactaram as previsões do modelo. Cada ponto representa um aluno, e o eixo X mostra o impacto daquela variável na previsão, valores positivos favorecem o "Reprovado". A cor indica o valor da variável, vermelho é alto e azul é baixo. A análise mostra que:
- **`Genero_Masculino`**: Ser do gênero masculino foi o fator que mais contribuiu para uma previsão de reprovação.
- **`Quiz3` e `Quiz1`**: Notas baixas nestes quizzes tiveram um forte impacto na previsão de risco de reprovação, enquanto notas altas foram fortes indicadores de aprovação.
- **`TempoQ1`**: Um tempo maior para completar o primeiro quiz também aumentou a chance de resultar na predição de uma reprovação.

**Interpretação da Performance**

&emsp; O modelo Nearest Centroid se mostrou uma ferramenta de alerta precoce altamente eficaz, alcançando um **excelente Recall de 80.8%**. Conforme a matriz de confusão, ele foi capaz de identificar corretamente **21 dos 26 alunos** que seriam reprovados, deixando apenas 5 casos passarem despercebidos (Falsos Negativos), cumprindo seu objetivo principal.

&emsp; Essa alta sensibilidade, no entanto, veio acompanhada de **51 Falsos Positivos**, resultando em uma precisão mais baixa e um **F1-Score de 42.9%**. O modelo, portanto, adota uma postura "cautelosa", preferindo gerar "alarmes falsos" a deixar um aluno em risco escapar. A análise SHAP enriquece essa conclusão, mostrando que essas previsões não são aleatórias, mas sim baseadas em fatores de risco claros e interpretáveis, como notas baixas nos primeiros quizzes e um tempo de resposta mais lento.

&emsp; Em resumo, o modelo da Semana 6 funciona como um sistema de alerta precoce robusto e transparente. Suas previsões, embora gerem um número considerável de falsos positivos, são ideais para acionar intervenções de baixo custo e amplo alcance, agora com a vantagem de sabermos exatamente quais fatores estão influenciando o alerta.

### 3.2 Análise Preditiva Intermediária (Semana 8)

&emsp; Avançando para a oitava semana, a análise do modelo definitivo entra em uma fase de maior maturidade. Com um conjunto de dados mais rico, que agora inclui avaliações mais robustas como a primeira prova parcial, o objetivo é transicionar de um "alerta precoce" para um **diagnóstico mais refinado**.

&emsp; A expectativa é que o **Nearest Centroid**, ao ser alimentado com essas informações mais consolidadas, aprimore sua capacidade de distinguir os alunos em risco, resultando em previsões mais precisas e acionáveis. A análise a seguir investigará tanto a evolução da performance do modelo quanto as mudanças nos fatores de risco mais influentes, novamente com o auxílio da interpretabilidade do SHAP.

In [ ]:
nearest_centroid(X2_train, X2_test, y2_train, y2_test)

#### Análise dos Resultados (Semana 8)

&emsp; Na análise intermediária da Semana 8, o `GridSearchCV` novamente selecionou a métrica de distância **'manhattan'** como a ideal, mantendo a mesma configuração de hiperparâmetros da análise anterior, o que sugere uma estabilidade na natureza dos dados.

&emsp; As métricas de avaliação para a classe de reprovados (classe 1) foram as seguintes:

- **Recall da classe 1 (Reprovado):** 0.769 (76.9%)
- **F1-Score da classe 1 (Reprovado):** 0.541 (54.1%)

**Interpretabilidade do Modelo com SHAP**

&emsp; A análise de interpretabilidade do SHAP revela uma mudança importante nos fatores de risco. A nota da **`Parcial_1`** surge como a segunda variável mais influente. Notas baixas (azul) nesta avaliação são agora um forte indicador de risco de reprovação, oferecendo ao modelo um sinal muito mais claro do que os quizzes isolados. `Genero_Masculino` continua sendo um fator de impacto relevante, e os quizzes anteriores mantêm seu padrão de influência (notas baixas aumentam o risco).

**Interpretação da Performance**

&emsp; A performance do modelo na Semana 8 mostra uma evolução clara em direção a um maior equilíbrio e confiabilidade. Embora o **Recall tenha tido uma leve queda para 76.9%**, identificando **20 dos 26** alunos em risco, o **F1-Score melhorou significativamente para 54.1%**.

&emsp; Essa melhora é explicada pela drástica redução no número de "alarmes falsos". A matriz de confusão mostra que o modelo gerou apenas **28 Falsos Positivos**, quase metade do número da Semana 6. O modelo sacrificou uma pequena parte de sua sensibilidade para se tornar muito mais preciso. Essa nova capacidade de discernimento é um reflexo direto da inclusão de dados mais robustos, como a `Parcial_1`, conforme indicado pelo SHAP.

&emsp; Em resumo, na Semana 8, o `Nearest Centroid` evoluiu de um sistema de alerta para uma ferramenta de diagnóstico mais confiável. Suas previsões são mais "acionáveis", justificando intervenções mais focadas no grupo de 48 alunos sinalizados (20 VPs + 28 FPs). Esta fase representa um excelente ponto de equilíbrio entre a antecedência da previsão e a assertividade do modelo.

### 3.3 Análise Preditiva Final (Semana 12)

&emsp; Chegando à etapa final da análise, o modelo definitivo `Nearest Centroid` é avaliado com o conjunto de dados mais completo. O objetivo desta seção é mensurar a performance máxima do modelo e obter um diagnóstico final e transparente sobre os fatores de risco.

&emsp; Com a totalidade das avaliações disponíveis, espera-se que o modelo atinja seu mais alto grau de equilíbrio A análise de performance e a interpretabilidade via SHAP irão compor o veredito final sobre a eficácia da solução proposta.

In [ ]:
nearest_centroid(X3_train, X3_test, y3_train, y3_test)

#### Análise dos Resultados (Semana 12)

&emsp; Na análise final da Semana 12, o `GridSearchCV` mais uma vez elegeu a métrica **'manhattan'** como a ideal, demonstrando a consistência do modelo e a robustez dessa métrica de distância para o conjunto de dados.

&emsp; As métricas de avaliação para a classe de reprovados (classe 1) foram as seguintes:

- **Recall da classe 1 (Reprovado):** 0.731 (73.1%)
- **F1-Score da classe 1 (Reprovado):** 0.691 (69.1%)

**Interpretabilidade do Modelo com SHAP**

&emsp; O gráfico SHAP da Semana 12 revela uma informação muito importante: o modelo aprendeu que o desempenho mais **recente** do aluno é o indicador mais forte de seu resultado final. As notas do **`Quiz5`** e **`Quiz6`** se tornaram as variáveis de maior impacto, onde notas baixas (azul) empurram a previsão fortemente para a reprovação. Fatores anteriormente dominantes, como a `Parcial_1` e `Genero_Masculino`, continuam relevantes, mas agora o modelo atribui um peso maior à trajetória final do estudante.

**Interpretação da Performance**

&emsp; A performance do modelo na Semana 12 representa o seu ápice de equilíbrio e eficácia. O **F1-Score atingiu um valor robusto de 69.1%**, indicando uma excelente harmonia entre a capacidade de detecção e a precisão das previsões. O **Recall de 73.1%** mostra que o modelo ainda é capaz de identificar a grande maioria dos alunos em risco, capturando **19 dos 26** casos de reprovação.

&emsp; O grande destaque desta fase final é a altíssima precisão do modelo. A matriz de confusão exibe um número mínimo de "alarmes falsos", com apenas **10 Falsos Positivos**. Isso significa que, quando o modelo sinaliza um aluno como "em risco" na Semana 12, ele está correto na maioria das vezes.

&emsp; Em sua conclusão, o modelo `Nearest Centroid` prova ser uma ferramenta de diagnóstico final altamente confiável e transparente. Ele não apenas alcança sua melhor performance, mas o faz baseando suas decisões nos fatores mais intuitivos e relevantes. O resultado é um modelo que não só prevê com acerto, mas cujas previsões são facilmente interpretáveis e justificáveis.

## 4. Conclusão

&emsp; O presente projeto teve como objetivo desenvolver um modelo de machine learning capaz de prever, em diferentes estágios do semestre, o risco de reprovação de alunos, com um foco especial na interpretabilidade e na clareza da solução. Para este fim, o classificador **Nearest Centroid** foi escolhido como modelo definitivo, devido à sua simplicidade, eficiência computacional e, principalmente, sua alta transparência.

&emsp; A análise demonstrou que o modelo não apenas foi eficaz, mas também exibiu uma evolução lógica e intuitiva ao longo do tempo. Na **Semana 6**, ele atuou como um "sistema de alerta precoce" de alta sensibilidade, ideal para acionar intervenções iniciais de baixo custo. Na **Semana 8**, com a inclusão de dados de avaliações mais robustas, o modelo amadureceu, tornando-se uma ferramenta de diagnóstico mais equilibrada e confiável. Finalmente, na **Semana 12**, o `Nearest Centroid` atingiu seu pico de performance, consolidando-se como uma ferramenta de diagnóstico final de alta precisão, com um **F1-Score de 69.1%** e um número muito baixo de falsos positivos.

&emsp; Um dos maiores sucessos do projeto foi a capacidade de interpretar as decisões do modelo em cada fase através da análise **SHAP**. Foi possível observar que o modelo aprendeu a dar mais peso aos indicadores de desempenho mais recentes e relevantes à medida que o semestre avançava. Essa interpretabilidade não apenas valida a lógica do modelo, mas também fornece um valor imenso para o contexto pedagógico, permitindo que as intervenções sejam baseadas em evidências claras e compreensíveis.

&emsp; Conclui-se, portanto, que o modelo `Nearest Centroid` se provou uma excelente escolha, entregando uma solução que é, ao mesmo tempo, robusta e transparente. Além de classificador, o projeto é uma ferramenta de diagnóstico evolutiva, capaz de fornecer informações preciosas ao parceiro acerca do desempenho dos alunos em determinado período do ano letivo.